In [ ]:
# -*- coding: utf-8 -*-
"""
================================================================================
  Graph RAG + LLM Legal Pipeline  —  v9.0  GRAPH RAG MULTI-MODEL BENCHMARK
================================================================================
  Doctoral Research Program · Predictive Model · Mexican Legal Sector (SCJN)
  Corpus     : SCJN Resolutions (PDF) + 9 Mexican Labor Law PDFs
  Benchmark  : gold_prediction_90_compatible.json  (90 reference cases)
  Rubric     : rubrica_validacion_contratos.json   (20 validation questions)
  Environment: Google Colab T4 GPU · Python 3.12 · spaCy 3.8

  OBJECTIVE
  ─────────
  Produce and evaluate a Spanish legal-recommendation assistant that generates
  a litigation STRATEGY for labor-contract cases under Mexican law, and identify
  — via a controlled benchmark — which base model best serves that objective
  inside a Graph RAG pipeline.

  EXPERIMENTAL DESIGN (controlled two-factor comparison)
  ──────────────────────────────────────────────────────
  The pipeline, prompts, generation budget and metrics are FROZEN across every
  run; the ONLY variable is MODEL_KEY (set in §2). Knowledge Graph is ALWAYS
  enabled — this is the Graph RAG arm. Five sequential runs:
    Factor A — ARCHITECTURE @ Q4_K_M (7B–9B band):
        llama31-8b-q4 · qwen25-7b-q4 · mistral-7b-v03-q4 · gemma2-9b-q4
    Factor B — QUANTIZATION @ Llama-3.1-8B (matched pair):
        llama31-8b-q4 (Q4_K_M)  vs  llama31-8b-q8 (Q8_0)
  ARM = f"GRAPH_RAG__{MODEL_KEY}" → each run writes its own artifact namespace.

  EVALUATION (3-tier metric hierarchy)
  ─────────────────────────────────────
  Tier 1 (headline)  : 3-class accuracy, macro-F1, MCC + per-class P/R/F1,
                       each with 95% bootstrap CIs (2000 resamples, seed=42).
  Tier 2 (grounding) : article precision / recall / F1 vs gold citations.
  Tier 3 (secondary) : ROUGE-1/2/L, BLEU, BERTScore-F1 (fluency proxy).
  Strategy quality   : 20-question rubric (LLM-as-judge, binary per criterion).
  Qualitative        : 5 representative cases for thesis narrative (§12b).

  PIPELINE (run once per MODEL_KEY, top to bottom):
    §0   ABI reset & kernel restart                   (auto)
    §1   Imports
    §PRE PRE-FLIGHT checks
    §2   Global constants + MODEL_REGISTRY
    §3   Model initialisation: GGUF → spaCy → Embeddings
    §4   Helper functions
    §5   PDF ingestion
    §6   RAG vector index (FAISS — shared across all 5 runs)
    §7   Knowledge Graph (cached on Drive — shared across all 5 runs)
    §8   Benchmark + Rubric loading
    §9   Inference loop (GPU-gated, checkpoint/resume)
    §10  Evaluation: Tier-1/2/3 metrics + bootstrap CIs
    §11  Litigation Strategy (structured RAG + KG output)
    §11b Rubric Evaluation (20-question LLM-as-judge)
    §12  Visualisation + §12b qualitative 5-case sample
    §13  Save all artifacts to Drive (ARM-namespaced)
    §14  Case Analysis Module (PhD experiment API)

  HOW TO RUN
  ──────────
  1. Place all 5 GGUF files in: /content/drive/MyDrive/SCJN_RAG/models/
  2. Set MODEL_KEY in §2 to the desired model (one run per key).
  3. Fresh Colab T4 session → run all cells top to bottom ONCE.
  4. §0 auto-restarts; sentinel detected → §1 continues.
  5. Repeat for each of the 5 MODEL_KEY values.
================================================================================
"""


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §0 · ENVIRONMENT SETUP  (v13 — replaces the legacy ABI-reset/kernel-restart)
#
# WHY THIS REPLACED THE OLD §0 (empirically established, not assumed):
#   The original §0 performed a full "ABI reset": it uninstalled numpy, scipy,
#   torchvision and torchaudio, force-pinned the stack back to
#   torch 2.5.1+cu121 / numpy<2.0, then killed the kernel to reload it. That was
#   correct for the Colab image of ~2024, whose default numpy/torch combination
#   broke llama-cpp-python and spaCy.
#
#   It is now actively harmful. Current Colab ships torch 2.11.0+cu128 with
#   torchvision/torchaudio matched and CUDA available, and every scientific
#   dependency imports cleanly out of the box. Forcing the old pins downgrades a
#   healthy environment into one that no longer matches the VM's CUDA driver,
#   which reproducibly caused: numpy reverting to 2.5.x (modern deps require
#   it), torchvision disappearing (uninstalled, then reinstalled against an
#   incompatible pin), and hard kernel deaths with NO Python traceback
#   (ABI-mismatched C extensions segfault the process — uncatchable).
#
#   VERIFIED: on a clean VM with §0 skipped entirely, §1 progressed past
#   torch/torchvision/numpy/spacy and failed ONLY on genuinely absent packages
#   (llama_cpp, then langchain_community). So this cell installs what is missing
#   and changes nothing that already works. No uninstalls, no pins, no restart.
#
# NOTE ON THE llama-cpp-python WHEEL: upstream publishes prebuilt CUDA wheels
#   for cu121-cu125/cu13x but NOT cu128, so we cannot index-match this VM
#   exactly. cu121 is used because CUDA minor-version compatibility lets a
#   12.1-built binary run on a 12.8 driver, and llama-cpp-python links the
#   SYSTEM CUDA runtime — it does not need to match torch's cu128.
#   --only-binary is deliberate: it forces a LOUD failure instead of silently
#   falling back to a CPU-only source build, which would make the 5x60-case
#   runs untenably slow.
# ──────────────────────────────────────────────────────────────────────────────
import subprocess, sys, importlib, os

print("§0  Environment setup (no ABI reset, no kernel restart)...")

_GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
# NOTE: downstream cells (§PRE, §3, §9) reference _HAS_GPU, which the legacy §0
# exported. Both names are bound here so no downstream cell needs editing.
_HAS_GPU = _GPU
_GPU_NAME = ""
if _HAS_GPU:
    try:
        _GPU_NAME = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True, text=True).stdout.strip().split("\n")[0]
    except Exception:
        _GPU_NAME = "GPU"
print(f"§0  GPU detected: {_GPU_NAME if _HAS_GPU else 'no — CPU runtime'}")

# ── 0.1  Pipeline packages (same set the legacy Phase A installed) ───────────
_PKGS = ["langchain", "langchain-community", "langchain-text-splitters",
         "sentence-transformers", "faiss-cpu", "pypdf", "PyPDF2",
         "evaluate", "rouge_score", "bert-score", "sacrebleu",
         "scikit-learn", "pandas", "matplotlib", "seaborn", "networkx", "spacy"]
print(f"§0  Installing {len(_PKGS)} pipeline packages (~2-3 min)...")
_r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_PKGS],
                    capture_output=True, text=True)
print("  ✅ pipeline packages installed" if _r.returncode == 0
      else f"  ⚠️  pip returned {_r.returncode}:\n{_r.stdout[-1200:]}\n{_r.stderr[-1200:]}")

# ── 0.2  llama-cpp-python (CUDA wheel; loud failure if unavailable) ──────────
try:
    import llama_cpp
    print(f"  ✅ llama_cpp already present ({llama_cpp.__version__})")
except ModuleNotFoundError:
    print("§0  Installing llama-cpp-python (prebuilt CUDA wheel)...")
    if _HAS_GPU:
        _w = subprocess.run(
            [sys.executable, "-m", "pip", "install", "llama-cpp-python",
             "--prefer-binary", "--only-binary=llama-cpp-python",
             "--extra-index-url",
             "https://abetlen.github.io/llama-cpp-python/whl/cu121"],
            capture_output=True, text=True)
        if _w.returncode != 0:
            print("  ⚠️  cu121 wheel unavailable → retrying cu125...")
            _w = subprocess.run(
                [sys.executable, "-m", "pip", "install", "llama-cpp-python",
                 "--prefer-binary", "--only-binary=llama-cpp-python",
                 "--extra-index-url",
                 "https://abetlen.github.io/llama-cpp-python/whl/cu125"],
                capture_output=True, text=True)
        print("  ✅ llama-cpp-python installed" if _w.returncode == 0
              else f"  ❌ FAILED — do NOT proceed (a CPU-only fallback would make "
                   f"the runs untenably slow):\n{_w.stdout[-1200:]}\n{_w.stderr[-1200:]}")
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "llama-cpp-python"], capture_output=True)

# ── 0.3  spaCy Spanish model (es_core_news_lg — KG is always on in v9+) ──────
_sp = subprocess.run([sys.executable, "-c",
                      "import spacy; spacy.load('es_core_news_lg'); print('ok')"],
                     capture_output=True, text=True)
if "ok" not in _sp.stdout:
    print("§0  Downloading spaCy es_core_news_lg (~600 MB)...")
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "es_core_news_lg", "--quiet"], capture_output=True)

# ── 0.4  Verify every §1 import target BEFORE handing off ────────────────────
print("\n§0  Verifying imports required by §1:")
_CHECK = ["torch", "torchvision", "torchaudio", "numpy", "pandas", "matplotlib",
          "seaborn", "networkx", "spacy", "llama_cpp", "langchain_community",
          "langchain_text_splitters", "evaluate", "bert_score", "sklearn"]
_missing = []
for _m in _CHECK:
    try:
        importlib.import_module(_m)
    except Exception as _e:
        _missing.append((_m, type(_e).__name__))
if _missing:
    print(f"  ❌ still missing/broken: {_missing}")
    print("  Do not run §1 yet — report this list.")
else:
    print(f"  ✅ all {len(_CHECK)} modules import cleanly — §1 is safe to run.")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §1  IMPORTS
# ──────────────────────────────────────────────────────────────────────────────
print("§1  Importing packages...")
import os, json, re, io, warnings, importlib, time as _time, fnmatch as _fnmatch
import shutil as _shutil_model, glob as _glob_model
warnings.filterwarnings("ignore")

# PyTorch — import order mandatory (I-26: prevents torchvision circular import)
import torch

# float8 compat shim: newer HF packages reference float8 dtypes absent in torch 2.5.1
# Alias to float32 so import-time lookups succeed; this pipeline never computes in fp8.
for _dt in ("float8_e8m0fnu","float8_e4m3fn","float8_e5m2",
            "float8_e4m3fnuz","float8_e5m2fnuz"):
    if not hasattr(torch, _dt):
        setattr(torch, _dt, torch.float32)

import torchvision          # noqa — must follow torch (I-26)
import torchaudio           # noqa
print(f"  torch {torch.__version__} | torchvision {torchvision.__version__} "
      f"| CUDA: {torch.cuda.is_available()}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import spacy
from spacy.pipeline import EntityRuler
from llama_cpp import Llama

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

import evaluate as hf_evaluate            # after torch is cached — safe here
from bert_score import score as bert_score_compute
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, matthews_corrcoef)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    _IN_COLAB = True
    print("  Google Drive mounted.")
except ImportError:
    _IN_COLAB = False
print("✅  Imports successful.\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §2  GLOBAL CONSTANTS + MODEL REGISTRY
# ──────────────────────────────────────────────────────────────────────────────

# ── Drive paths ───────────────────────────────────────────────────────────────
DRIVE_ROOT       = "/content/drive/MyDrive/SCJN_RAG"
MODELS_DIR_DRIVE = os.path.join(DRIVE_ROOT, "models")
MODELS_DIR_LOCAL = "/content/models"
RAW_PDFS_PATH    = os.path.join(DRIVE_ROOT, "data/scjn_resoluciones")
LAW_PDFS_PATH    = os.path.join(DRIVE_ROOT, "data/leyes_mexicanas")
EVAL_JSON_PATH   = os.path.join(DRIVE_ROOT, "data/eval_json")
FAISS_INDEX_PATH = os.path.join(DRIVE_ROOT, "faiss_index")
KG_DRIVE_PATH    = os.path.join(DRIVE_ROOT, "legal_kg.graphml")
RUBRIC_JSON_PATH = os.path.join(DRIVE_ROOT, "data/eval_json",
                                "rubrica_validacion_contratos.json")
LOCAL_RAW_PDFS   = "/content/raw_pdfs"
LOCAL_LAW_PDFS   = "/content/law_pdfs"
LOCAL_JSON       = "/content/test_cases_json"

# ── Experimental flag — always True in v9 (Graph RAG arm) ────────────────────
KG_ENABLED = True     # this script IS the Graph RAG arm; never set to False here

# ── MODEL REGISTRY — controlled two-factor benchmark ─────────────────────────
# pattern : case-insensitive glob to locate the GGUF (exact quant token required)
# n_ctx   : None → use global N_CTX; int → per-model override (VRAM safety)
MODEL_REGISTRY = {
    # pattern              : case-insensitive glob for GGUF resolution
    # n_ctx                : None → global N_CTX; int → per-model VRAM-safe override
    # supports_system_role : True for all models except Gemma-2, which has no
    #                        <system> token in its chat template (Google design choice).
    #                        When False, _build_messages() folds the system instruction
    #                        into the user turn as a prefix (I-40).

    # Factor A — architecture @ Q4_K_M (7B–9B band)
    "llama31-8b-q4":     {"pattern":"*Llama-3.1-8B-Instruct-Q4_K_M*.gguf",
                          "arch":"Llama-3.1","params_b":8,"quant":"Q4_K_M",
                          "chat_format":None,"reasoning":False,"n_ctx":None,
                          "supports_system_role":True},
    "qwen25-7b-q4":      {"pattern":"*Qwen2.5-7B-Instruct*Q4_K_M*.gguf",
                          "arch":"Qwen2.5","params_b":7,"quant":"Q4_K_M",
                          "chat_format":None,"reasoning":False,"n_ctx":None,
                          "supports_system_role":True},
    "mistral-7b-v03-q4": {"pattern":"*Mistral-7B-Instruct-v0.3*Q4_K_M*.gguf",
                          "arch":"Mistral-v0.3","params_b":7,"quant":"Q4_K_M",
                          "chat_format":None,"reasoning":False,"n_ctx":None,
                          "supports_system_role":True},
    "gemma2-9b-q4":      {"pattern":"*gemma-2-9b-it*Q4_K_M*.gguf",
                          "arch":"Gemma-2","params_b":9,"quant":"Q4_K_M",
                          "chat_format":None,"reasoning":False,"n_ctx":4096,
                          "supports_system_role":False},   # I-40: no <system> token
    # Factor B — quantization @ Llama-3.1-8B (matched pair with llama31-8b-q4)
    "llama31-8b-q8":     {"pattern":"*Llama-3.1-8B-Instruct-Q8_0*.gguf",
                          "arch":"Llama-3.1","params_b":8,"quant":"Q8_0",
                          "chat_format":None,"reasoning":False,"n_ctx":4096,
                          "supports_system_role":True},
}

# ►►►  SET THE MODEL FOR THIS RUN (one per key; 5 runs total)  ◄◄◄
MODEL_KEY = "llama31-8b-q4"
assert MODEL_KEY in MODEL_REGISTRY, \
    f"MODEL_KEY '{MODEL_KEY}' not in {list(MODEL_REGISTRY)}"

MODEL_SPEC      = MODEL_REGISTRY[MODEL_KEY]
MODEL_ARCH      = MODEL_SPEC["arch"]
MODEL_PARAMS_B  = MODEL_SPEC["params_b"]
MODEL_QUANT     = MODEL_SPEC["quant"]
MODEL_REASONING       = MODEL_SPEC.get("reasoning", False)
MODEL_SUPPORTS_SYSTEM = MODEL_SPEC.get("supports_system_role", True)   # I-40: Gemma-2 has no system role in its chat template
N_CTX                 = MODEL_SPEC.get("n_ctx") or 8192   # per-model override or global

# ── JUDGE — fixed, independent, never a benchmark candidate ──────────────────
# CORRECTIVE FIX (post-hoc methodological review): this arm's rubric
# evaluation previously reused the CANDIDATE model as its own judge (self-
# grading), which introduces self-preference / self-enhancement bias in
# LLM-as-judge protocols (Zheng et al., 2023, "Judging LLM-as-a-Judge with
# MT-Bench and Chatbot Arena", NeurIPS) and breaks comparability with the
# RL+LLM arm, which uses a fixed, out-of-family judge (Phi-3.5-mini-instruct).
# This block brings the arm into methodological alignment: ALL arms score the
# 20-question rubric with the SAME fixed, independent judge — never the
# candidate under test.
JUDGE_KEY  = "phi35-mini-q4"
JUDGE_SPEC = {"pattern": "*Phi-3.5-mini-instruct*Q4_K_M*.gguf",
              "arch": "Phi-3.5", "params_b": 3.8, "quant": "Q4_K_M",
              "chat_format": None, "n_ctx": 4096, "supports_system": True,
              # CPU JUDGE (n_gpu_layers=0). Two reasons, both load-bearing:
              #  1. VRAM. The judge would otherwise be loaded with all layers on
              #     GPU while the candidate is resident. With the Q8_0 candidate
              #     (~8.5 GB) on a 16 GB accelerator this exhausts memory and
              #     llama.cpp returns NULL, surfacing as the misleading message
              #     "Failed to load model from file" — a memory failure wearing
              #     the costume of a missing-file failure.
              #  2. Measurement consistency. The judge is the measuring
              #     instrument for all 20 conditions; its execution environment
              #     must not vary across the matrix.
              "n_gpu_layers": 0}
assert MODEL_KEY != JUDGE_KEY, (
    "MODEL_KEY cannot equal JUDGE_KEY — the judge must stay out of the "
    "benchmark loop to avoid self-enhancement bias (Zheng et al., 2023).")

# ARM naming: GRAPH_RAG__{MODEL_KEY} — distinguishes from RAG-only arm in comparison
ARM        = f"GRAPH_RAG__{MODEL_KEY}"
ARM_SUFFIX = f"graphrag_{MODEL_KEY}"

# ── Shared pipeline invariants (identical across all 5 runs) ─────────────────
DECISION_LABELS   = ["Procedente","Improcedente","Parcialmente Improcedente"]
DECISION_LABEL2ID = {v: i for i, v in enumerate(DECISION_LABELS)}

LEGAL_ENTITY_TYPES = ["PER","ORG","LOC","MISC","DATE","GPE",
                       "LAW_ARTICLE","LEGAL_CONCEPT"]
SPANISH_VERB_TO_RELATION = {
    "despedir":"RESCINDS","rescindir":"RESCINDS",
    "demandar":"DEMANDS","regular":"REGULATES",
    "establecer":"ESTABLISHES","violar":"VIOLATES",
    "citar":"CITES","involucrar":"INVOLVES",
}

CHUNK_SIZE_PREC    = 2000;  CHUNK_OVERLAP_PREC = 200
CHUNK_SIZE_LAW     = 2400;  CHUNK_OVERLAP_LAW  = 300
RETRIEVER_K        = 4

# ── PATCH A: GPU gate threshold ───────────────────────────────────────────────
MIN_TPS = 8.0   # abort §9 if probe < 8 tok/s (CPU ≈ 0.8, T4 GPU ≈ 30+)

# ── PATCH B: checkpoint / resume ─────────────────────────────────────────────
RESUME           = True
CHECKPOINT_LOCAL = f"/content/checkpoint_{ARM_SUFFIX}.jsonl"
CHECKPOINT_DRIVE = os.path.join(DRIVE_ROOT,"results",ARM,
                                f"checkpoint_{ARM_SUFFIX}.jsonl")

print(f"§2  Model={MODEL_KEY} | arch={MODEL_ARCH} {MODEL_PARAMS_B}B | "
      f"quant={MODEL_QUANT} | n_ctx={N_CTX} | KG={KG_ENABLED} | arm={ARM}")

# ══════════════════════════════════════════════════════════════════════════
# CANONICAL ARTICLE NORMALIZER — single source of truth on Drive
# ══════════════════════════════════════════════════════════════════════════
# Legal article citations appear in incompatible surface forms across the gold
# set and the arms:
#     gold      "LFT arts. 20–21"   "CPEUM art. 123-A"
#     RAG/GRAG  "20 Ley Federal del Trabajo (LFT)"
#     RL        "LFT art. 12"
# Comparing these as raw strings (only .strip().lower()) scores a correct
# citation as a miss purely because of formatting. Measured impact on the
# 60-case draft: article F1 of 0.005 / 0.000 / 0.048 raw, versus 0.134 / 0.159
# / 0.207 canonicalized — i.e. the raw numbers were an artifact of output
# format, not of legal grounding.
#
# norm.py lives ONCE on Drive and is imported by ALL FOUR arms and by
# CrossModelCompar. A per-notebook copy would drift and produce two different
# article scores for the same run with no visible cause. Missing module is a
# HARD FAILURE — a silent fallback to raw matching is exactly the bug this
# module exists to remove.
_LIB_DIR = os.path.join(DRIVE_ROOT, "lib")
if _LIB_DIR not in sys.path:
    sys.path.insert(0, _LIB_DIR)
try:
    from norm import normalize_articles
except ModuleNotFoundError as _e:
    raise ModuleNotFoundError(
        f"Canonical article normalizer not found: {_LIB_DIR}/norm.py\n"
        f"  Upload norm.py to {_LIB_DIR}/ before running. Article grounding "
        f"metrics are invalid without it (raw string matching scores correct "
        f"citations as misses whenever the surface form differs from gold)."
    ) from _e
print(f"  ✅ canonical article normalizer loaded from {_LIB_DIR}/norm.py")

try:
    from index_manifest import ensure_manifest
except ModuleNotFoundError as _e:
    raise ModuleNotFoundError(
        f"index_manifest.py not found in {_LIB_DIR}/ — required to verify that "
        f"this arm retrieves from the same frozen index as the others."
    ) from _e


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §PRE  PRE-FLIGHT CHECKS (before loading any model)
# ──────────────────────────────────────────────────────────────────────────────
print("\n── §PRE  Pre-flight checks ────────────────────────────────────────────")
_pf_ok = True

def _pf(label, cond, fix=""):
    global _pf_ok
    if cond:
        print(f"  ✅  {label}")
    else:
        print(f"  ❌  {label}")
        if fix: print(f"       Fix: {fix}")
        _pf_ok = False

# 1. Drive mounted
_pf("Drive mounted",
    os.path.isdir(DRIVE_ROOT),
    "Run drive.mount('/content/drive') or enable Drive in Colab.")

# 2. GGUF file resolvable
_pat = MODEL_SPEC["pattern"].lower()
_gguf_hits = []
for _d in [MODELS_DIR_LOCAL, MODELS_DIR_DRIVE, "/content", DRIVE_ROOT]:
    if os.path.isdir(_d):
        for _fn in os.listdir(_d):
            if _fn.lower().endswith(".gguf") and _fnmatch.fnmatch(_fn.lower(), _pat):
                _gguf_hits.append(os.path.join(_d, _fn))
_pf(f"GGUF '{MODEL_SPEC['pattern']}' found",
    bool(_gguf_hits),
    f"Place the GGUF in {MODELS_DIR_DRIVE}/")

# 3. Benchmark JSON
_gold = os.path.join(EVAL_JSON_PATH,"gold_prediction_90_compatible.json")
_pf("Benchmark JSON (gold_prediction_90_compatible.json) found",
    os.path.isfile(_gold),
    f"Upload to {EVAL_JSON_PATH}/")

# 4. Rubric JSON
_pf("Rubric JSON (rubrica_validacion_contratos.json) found",
    os.path.isfile(RUBRIC_JSON_PATH),
    f"Upload to {os.path.dirname(RUBRIC_JSON_PATH)}/")

# 5. Free VRAM estimate
if _HAS_GPU:
    _vmem = subprocess.run(
        ["nvidia-smi","--query-gpu=memory.free","--format=csv,noheader,nounits"],
        capture_output=True, text=True)
    try:
        _free_mb = int(_vmem.stdout.strip().split("\n")[0])
        _free_gb = _free_mb / 1024
        _need_gb = MODEL_PARAMS_B * 0.65 + 1.5   # rough: params×0.65 + e5-large buffer
        _pf(f"Free VRAM ≥ {_need_gb:.1f} GB (have {_free_gb:.1f} GB)",
            _free_gb >= _need_gb,
            "Disconnect and delete runtime to free VRAM.")
    except Exception:
        print("  ⚠️  Could not parse VRAM — proceeding.")

if not _pf_ok:
    raise RuntimeError("❌  Pre-flight failed. Fix the issues above before continuing.")
print("✅  All pre-flight checks passed.\n")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §3  MODEL INITIALISATION
# ──────────────────────────────────────────────────────────────────────────────

# ── 3.1  Resolve and load GGUF ────────────────────────────────────────────────
def _resolve_gguf(pattern, search_dirs):
    """Case-insensitive first-match GGUF resolver across an ordered directory list."""
    pat = pattern.lower()
    hits = []
    for d in search_dirs:
        if os.path.isdir(d):
            for fn in os.listdir(d):
                if fn.lower().endswith(".gguf") and _fnmatch.fnmatch(fn.lower(), pat):
                    hits.append(os.path.join(d, fn))
    return sorted(set(hits))

_SEARCH_DIRS = [MODELS_DIR_LOCAL, MODELS_DIR_DRIVE, "/content", DRIVE_ROOT]

print(f"§3.1  Loading model '{MODEL_KEY}'  (pattern: {MODEL_SPEC['pattern']})...")
_matches = _resolve_gguf(MODEL_SPEC["pattern"], _SEARCH_DIRS)
if not _matches:
    _avail = []
    for _d in _SEARCH_DIRS:
        if os.path.isdir(_d):
            _avail += [os.path.basename(p)
                       for p in _resolve_gguf("*.gguf",[_d])]
    raise FileNotFoundError(
        f"No GGUF matching '{MODEL_SPEC['pattern']}' for MODEL_KEY='{MODEL_KEY}'.\n"
        f"  Searched: {_SEARCH_DIRS}\n"
        f"  .gguf files found: {sorted(set(_avail)) or '(none)'}\n"
        f"  → Place the file in {MODELS_DIR_DRIVE}/")

if len(_matches) > 1:
    print(f"  ⚠️  {len(_matches)} files match — using first: {os.path.basename(_matches[0])}")
_resolved = _matches[0]
print(f"  Resolved: {_resolved}")

# Cache to local SSD for faster loading
_LOCAL_MODEL = os.path.join("/content", os.path.basename(_resolved))
if _resolved.startswith("/content"):
    _model_path, _src = _resolved, "local_ssd"
elif os.path.exists(_LOCAL_MODEL):
    _model_path, _src = _LOCAL_MODEL, "local_ssd"
else:
    print("  Caching model to local SSD...")
    try:
        _shutil_model.copy2(_resolved, _LOCAL_MODEL)
        _model_path, _src = _LOCAL_MODEL, "drive_cache"
    except Exception as _e:
        print(f"  ⚠️  Local cache failed ({_e}) → loading directly from Drive.")
        _model_path, _src = _resolved, "drive_direct"

_llama_kwargs = dict(model_path=_model_path, n_ctx=N_CTX,
                     n_gpu_layers=-1, verbose=False)
if MODEL_SPEC.get("chat_format"):
    _llama_kwargs["chat_format"] = MODEL_SPEC["chat_format"]

try:
    llm = Llama(**_llama_kwargs)
except Exception as _e:
    raise RuntimeError(
        f"Failed to load '{MODEL_KEY}': {_e}\n"
        f"  If 'unknown architecture': delete sentinel, re-run §0 for a newer wheel,\n"
        f"  or set an explicit chat_format in MODEL_REGISTRY (§2).")

print(f"  Source: {_src} | chat_format: {MODEL_SPEC.get('chat_format') or 'auto'} "
      f"| n_ctx={N_CTX}")

# Speed probe — quick check before §9 (hard gate happens in §9 via verify_inference_speed)
_t0 = _time.time()
_p  = llm.create_chat_completion(messages=[{"role":"user","content":"Di 'Hola'."}],
                                 max_tokens=5, temperature=0.0)
_rate = _p.get("usage",{}).get("completion_tokens",5) / max(_time.time()-_t0, 1e-6)
print(f"✅  Model ready (~{_rate:.0f} tok/s) | "
      f"Est. §9 ETA: ~{60*(512/max(_rate,0.1))/60:.0f} min")

# ══════════════════════════════════════════════════════════════════════════
# §3.1b · FIXED JUDGE MODEL — Phi-3.5-mini-instruct (independent of candidate)
# The judge loads ON DEMAND (only around rubric scoring) and releases
# immediately after — never co-resident with the candidate during
# generation, which is where peak VRAM pressure actually occurs. This
# matters most for the largest candidate (llama31-8b-q8, Q8_0): permanent
# co-residency of candidate + judge + embeddings model on a single T4 risks
# a hard CUDA OOM abort — the same failure class already observed on this
# model in the RL+LLM arm. Mirrors the LazyLLM pattern validated there.
# ══════════════════════════════════════════════════════════════════════════
class LazyLLM:
    """Loads the fixed judge GGUF on first use; releases on demand."""
    def __init__(self, model_key, spec, search_dirs):
        self.model_key, self.spec, self.search_dirs = model_key, spec, search_dirs
        self._llm = None

    @property
    def is_loaded(self):
        return self._llm is not None

    def get(self):
        if self._llm is None:
            print(f"  Loading fixed judge '{self.model_key}' (on-demand, "
                  f"n_gpu_layers={self.spec.get('n_gpu_layers', -1)})...")
            _matches = _resolve_gguf(self.spec["pattern"], self.search_dirs)
            if not _matches:
                raise FileNotFoundError(
                    f"No GGUF matching '{self.spec['pattern']}' for "
                    f"JUDGE_KEY='{self.model_key}'.\n  Searched: {self.search_dirs}\n"
                    f"  Place Phi-3.5-mini-instruct-Q4_K_M.gguf in {MODELS_DIR_DRIVE}/ "
                    f"before re-running — rubric scoring cannot proceed without a "
                    f"fixed, independent judge (see corrective methodology note in §2).")
            _resolved = _matches[0]
            _local = os.path.join("/content", os.path.basename(_resolved))
            if _resolved.startswith("/content") and not _resolved.startswith("/content/drive"):
                _path = _resolved
            elif os.path.exists(_local):
                _path = _local
            else:
                try:
                    _shutil_model.copy2(_resolved, _local); _path = _local
                except Exception as _e:
                    print(f"  ⚠️  Judge local cache failed ({_e}) → loading directly from source.")
                    _path = _resolved
            # FIX: honour the spec instead of hardcoding full GPU offload.
            _ngl = self.spec.get("n_gpu_layers", -1)
            _kwargs = dict(model_path=_path, n_ctx=self.spec["n_ctx"],
                           n_gpu_layers=_ngl, verbose=False)
            if self.spec.get("chat_format"):
                _kwargs["chat_format"] = self.spec["chat_format"]
            try:
                self._llm = Llama(**_kwargs)
            except Exception as _e:
                raise RuntimeError(
                    f"Failed to load fixed judge '{self.model_key}': {_e}\n"
                    f"  Rubric scoring cannot proceed without an independent judge. "
                    f"If VRAM is the issue, lower JUDGE_SPEC['n_ctx'] in §2.")
            print(f"  Judge loaded from: {_path} (n_ctx={self.spec['n_ctx']})")
        return self._llm

    def release(self):
        if self._llm is not None:
            del self._llm
            self._llm = None
            gc.collect()
            if torch.cuda.is_available():
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass
            print(f"  Judge '{self.model_key}' released (VRAM freed).")

import gc
judge_mgr = LazyLLM(JUDGE_KEY, JUDGE_SPEC, _SEARCH_DIRS)
print(f"§3.1b  Fixed judge '{JUDGE_KEY}' configured for ON-DEMAND loading "
      f"(loads only around rubric scoring, never held resident during generation).")


# ── 3.2  spaCy NLP + EntityRuler (KG_ENABLED=True always in v9) ──────────────
def _ensure_spacy_model(name: str):
    """Load spaCy model; install in-process if absent."""
    try:
        return spacy.load(name)
    except OSError:
        pass
    print(f"  Installing {name} in current process...")
    for cmd in [[sys.executable,"-m","spacy","download",name,"--quiet"],
                [sys.executable,"-m","pip","install","-q",
                 name.replace("_","-")]]:
        subprocess.run(cmd, capture_output=True)
        importlib.invalidate_caches()
        try:
            return spacy.load(name)
        except OSError:
            continue
    raise RuntimeError(f"Cannot load '{name}'. Run: !python -m spacy download {name}")

print("§3.2  Loading spaCy es_core_news_lg...")
nlp = _ensure_spacy_model("es_core_news_lg")
nlp.max_length = 1_500_000

# Guard: remove ruler if present (prevents [E007] on double §3 run — I-31)
if "entity_ruler" in nlp.pipe_names:
    nlp.remove_pipe("entity_ruler")
if "ner" not in nlp.pipe_names:
    raise RuntimeError(
        f"NER missing from pipeline. Got: {nlp.pipe_names}\n"
        f"Run: !python -m spacy download es_core_news_lg")

# EntityRuler AFTER ner with overwrite_ents=True (I-29 fix)
ruler = nlp.add_pipe("entity_ruler", after="ner", config={"overwrite_ents": True})

_LAW_ABBREVS    = ["LFT","CPEUM","LSS","IMSS","INFONAVIT","STPS"]
_JURIS_PREFIXES = ["1a","2a","3a","1o","2o","3o"]

_law_pats = [
    {"label":"LAW_ARTICLE","pattern":[{"LOWER":{"IN":["artículo","articulo"]}},
     {"IS_DIGIT":True},{"LOWER":{"IN":["-","al","y"]},"OP":"?"},
     {"IS_DIGIT":True,"OP":"?"}]},
    {"label":"LAW_ARTICLE","pattern":[{"LOWER":{"IN":["art","arts"]}},
     {"ORTH":"."},{"IS_DIGIT":True},
     {"LOWER":{"IN":["-","al","y"]},"OP":"?"},{"IS_DIGIT":True,"OP":"?"}]},
]
for _abbr in _LAW_ABBREVS:
    _law_pats.append({"label":"LAW_ARTICLE",
                      "pattern":[{"ORTH":_abbr},{"IS_DIGIT":True}]})
    _law_pats.append({"label":"LAW_ARTICLE",
                      "pattern":[{"ORTH":_abbr},
                                 {"LOWER":{"IN":["art","arts","artículo","articulo"]}},
                                 {"ORTH":".","OP":"?"},{"IS_DIGIT":True}]})
for _pfx in _JURIS_PREFIXES:
    for _sala in ["J","PC"]:
        _law_pats.append({"label":"LAW_ARTICLE",
                          "pattern":[{"LOWER":_pfx},{"ORTH":"."},{"ORTH":"/"},
                                     {"ORTH":_sala},{"ORTH":"."},{"IS_DIGIT":True},
                                     {"ORTH":"/"},{"IS_DIGIT":True}]})
_concept_pats = [
    {"label":"LEGAL_CONCEPT","pattern":[{"LOWER":"despido"},
     {"LOWER":{"IN":["injustificado","justificado"]}}]},
    {"label":"LEGAL_CONCEPT","pattern":[{"LOWER":"rescisión"},
     {"LOWER":"contractual","OP":"?"}]},
    {"label":"LEGAL_CONCEPT","pattern":[{"LOWER":"relación"},{"LOWER":"laboral"}]},
    {"label":"LEGAL_CONCEPT","pattern":[{"LOWER":"indemnización"},
     {"LOWER":"constitucional"}]},
    {"label":"LEGAL_CONCEPT","pattern":[{"LOWER":"prima"},{"LOWER":"de"},
     {"LOWER":"antigüedad"}]},
    {"label":"LEGAL_CONCEPT","pattern":[{"LOWER":"salarios"},{"LOWER":"caídos"}]},
    {"label":"LEGAL_CONCEPT","pattern":[{"LOWER":"aviso"},{"LOWER":"de"},
     {"LOWER":"rescisión"}]},
    {"label":"LEGAL_CONCEPT","pattern":[{"LOWER":"subordinación"}]},
    {"label":"LEGAL_CONCEPT","pattern":[{"LOWER":"principio"},{"LOWER":"de"},
     {"LOWER":"primacía"},{"LOWER":"de"},{"LOWER":"la"},{"LOWER":"realidad"}]},
    {"label":"LEGAL_CONCEPT","pattern":[{"LOWER":"presunción"},{"LOWER":"de"},
     {"LOWER":"laboralidad"}]},
    {"label":"LEGAL_CONCEPT","pattern":[{"LOWER":"contrato"},
     {"LOWER":"colectivo","OP":"?"},{"LOWER":"de"},{"LOWER":"trabajo"}]},
    {"label":"LEGAL_CONCEPT","pattern":[{"LOWER":"jornada"},{"LOWER":"laboral"}]},
]
ruler.add_patterns(_law_pats + _concept_pats)

_smoke = nlp("Juan García trabajó en la empresa Constructora del Norte S.A. en México.")
_smoke_ents = [(e.text,e.label_) for e in _smoke.ents]
if not _smoke_ents:
    raise RuntimeError(
        "NER smoke test returned 0 entities — pipeline misconfigured.\n"
        f"  pipe_names: {nlp.pipe_names} | ruler.overwrite: {ruler.overwrite}\n"
        "  Restart kernel and re-run all cells from §0.")
print(f"✅  spaCy ready | pipes: {nlp.pipe_names}")
print(f"    Smoke test: {_smoke_ents[:3]}")

# ── 3.3  Sentence embeddings ──────────────────────────────────────────────────
_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n§3.3  Loading multilingual-e5-large on '{_DEVICE}'...")
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={"device":_DEVICE},
    encode_kwargs={"normalize_embeddings":True,"batch_size":64})
print(f"✅  Embeddings ready (device={_DEVICE}, batch_size=64).\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §4  HELPER FUNCTIONS
# ──────────────────────────────────────────────────────────────────────────────

# ── 4.1  PDF loader ──────────────────────────────────────────────────────────
def load_pdfs_from_directory(directory_path: str) -> list:
    docs = []
    if not os.path.isdir(directory_path):
        print(f"  ⚠️  Not found: {directory_path}"); return docs
    for fn in sorted(os.listdir(directory_path)):
        if fn.lower().endswith(".pdf"):
            try:
                docs.extend(PyPDFLoader(os.path.join(directory_path,fn)).load())
            except Exception as e:
                print(f"    ❌  {fn}: {e}")
    print(f"  {len(docs)} pages from '{directory_path}'.")
    return docs

# ── 4.2  Gold benchmark loader ────────────────────────────────────────────────
def load_test_cases_gold(json_path: str) -> list:
    def _parse(data):
        if "cases" not in data: return []
        out = []
        for c in data["cases"]:
            gs=c.get("gold_standard",{}); cel=c.get("expected_cel_output",{})
            bm=c.get("benchmark_metadata",{})
            out.append({"case_id":c.get("case_id","UNKNOWN"),
                "query":(c.get("input_summary","")+"\n\n"+c.get("facts","")).strip(),
                "ground_truth":"\n".join(cel.get("conclusiones_legales",[])),
                "gold_decision":gs.get("decision",""),
                "gold_articles":cel.get("articulos_citados",[]),
                "confidence":gs.get("confidence",""),
                "difficulty":bm.get("difficulty",""),
                "category":bm.get("category",""),
                "critical":bm.get("critical",False),
                "precedent_refs":gs.get("precedent_refs",[])})
        return out
    if os.path.isfile(json_path):
        with open(json_path,"r",encoding="utf-8") as f:
            cases = _parse(json.load(f))
        print(f"  Loaded {len(cases)} cases from '{json_path}'."); return cases
    if os.path.isdir(json_path):
        cases = []
        for jf in sorted(os.path.join(json_path,f)
                         for f in os.listdir(json_path)
                         if f.lower().endswith(".json")):
            try:
                with open(jf,"r",encoding="utf-8") as f:
                    cases.extend(_parse(json.load(f)))
            except Exception as e:
                print(f"    ❌  {jf}: {e}")
        print(f"  Loaded {len(cases)} cases total."); return cases
    return []

# ── 4.3  Text sanitiser ───────────────────────────────────────────────────────
def sanitize_text(text: str, max_chars: int = 1_400_000) -> str:
    """Remove PDF Unicode artifacts that crash spaCy's Cython tokenizer."""
    import unicodedata as _ud
    if not text or not isinstance(text,str): return ""
    try:
        text = text.encode("utf-16","surrogatepass").decode("utf-16","ignore")
        text = text.encode("utf-8","ignore").decode("utf-8","ignore")
    except Exception:
        text = text.encode("ascii","ignore").decode("ascii")
    _SAFE={"\t","\n","\r"}; buf=[]
    for ch in text:
        cat=_ud.category(ch)
        if cat in("Cs","Co"): continue
        if cat=="Cc" and ch not in _SAFE: continue
        if cat=="Cf" and ord(ch)!=0x00AD: continue
        buf.append(ch)
    text="".join(buf)
    text=text.replace("\x0c","\n")
    text=re.sub(r"[ \t]+"," ",text); text=re.sub(r"\n{3,}","\n\n",text)
    if len(text)>max_chars:
        cut=text[:max_chars].rfind(".")
        text=text[:cut+1] if cut>max_chars//2 else text[:max_chars]
    return text.strip()

# ── 4.4  Entity & relation extraction ─────────────────────────────────────────
def extract_entities_and_relationships(text: str, nlp_model) -> dict:
    text = sanitize_text(text)
    if not text: return {"entities":[],"relationships":[]}
    doc = nlp_model(text)
    entities,seen=[],set()
    for ent in doc.ents:
        if ent.label_ in LEGAL_ENTITY_TYPES and ent.text not in seen:
            entities.append({"text":ent.text,"label":ent.label_,
                             "start":ent.start_char,"end":ent.end_char})
            seen.add(ent.text)
    relationships,seen_rels=[],set()
    for sent in doc.sents:
        ents_in=[e for e in doc.ents if e.sent==sent
                 and e.label_ in LEGAL_ENTITY_TYPES]
        for i,e1 in enumerate(ents_in):
            for j,e2 in enumerate(ents_in):
                if i==j: continue
                for tok in sent:
                    if tok.pos_=="VERB" and e1.end_char<tok.idx<e2.start_char:
                        rel=SPANISH_VERB_TO_RELATION.get(tok.lemma_.lower(),
                                                          tok.lemma_.upper())
                        key=(e1.text,rel,e2.text)
                        if key not in seen_rels:
                            relationships.append({"head":e1.text,"type":rel,"tail":e2.text})
                            seen_rels.add(key)
        for tok in sent:
            if tok.lemma_.lower() not in SPANISH_VERB_TO_RELATION: continue
            subj=next((c for c in tok.children if c.dep_ in("nsubj","nsubj:pass")),None)
            obj=next((c for c in tok.children if c.dep_ in("obj","dobj")),None)
            if not(subj and obj): continue
            s_e=next((e["text"] for e in entities if subj.text in e["text"]),None)
            o_e=next((e["text"] for e in entities if obj.text  in e["text"]),None)
            if s_e and o_e:
                rel=SPANISH_VERB_TO_RELATION[tok.lemma_.lower()]
                key=(s_e,rel,o_e)
                if key not in seen_rels:
                    relationships.append({"head":s_e,"type":rel,"tail":o_e})
                    seen_rels.add(key)
    return {"entities":entities,"relationships":relationships}

# ── 4.5  KG builder ───────────────────────────────────────────────────────────
def build_knowledge_graph(entities: list, relationships: list) -> nx.DiGraph:
    G=nx.DiGraph()
    for e in entities:
        if not G.has_node(e["text"]): G.add_node(e["text"],label=e["label"])
    for r in relationships:
        for n,lbl in[(r["head"],"UNKNOWN"),(r["tail"],"UNKNOWN")]:
            if not G.has_node(n): G.add_node(n,label=lbl)
        G.add_edge(r["head"],r["tail"],type=r["type"])
    return G

# ── 4.6  Graph-aware retriever ────────────────────────────────────────────────
def graph_aware_retriever(query: str, graph, nlp_model) -> dict:
    if graph is None or nlp_model is None: return {"nodes":[],"edges":[]}
    doc=nlp_model(query)
    q_ents=[e.text for e in doc.ents if e.label_ in LEGAL_ENTITY_TYPES]
    nodes,edges,sn,se=[],[],set(),set()
    for qe in q_ents:
        matches=[n for n in graph.nodes if qe.lower() in n.lower()]
        for n in matches:
            if n not in sn:
                nodes.append({"text":n,"label":graph.nodes[n].get("label","N/A")}); sn.add(n)
            for nb in list(graph.neighbors(n))+list(graph.predecessors(n)):
                if nb not in sn:
                    nodes.append({"text":nb,"label":graph.nodes[nb].get("label","N/A")}); sn.add(nb)
                ed=graph.get_edge_data(n,nb) or graph.get_edge_data(nb,n)
                if ed:
                    key=(n,nb,ed.get("type"))
                    if key not in se:
                        edges.append({"head":n,"type":ed.get("type"),"tail":nb}); se.add(key)
    return {"nodes":nodes,"edges":edges}

# ── 4.7  Format graph context ─────────────────────────────────────────────────
def format_graph_context_for_llm(query: str, ctx: dict) -> str:
    if not ctx or(not ctx.get("nodes") and not ctx.get("edges")): return ""
    parts=[f"Consulta: {query}","--- Grafo de Conocimiento ---"]
    parts+=[f"  Entidad: {n['text']} (Tipo: {n['label']})" for n in ctx.get("nodes",[])]
    parts+=[f"  Relación: ({e['head']}) -[{e['type']}]-> ({e['tail']})"
            for e in ctx.get("edges",[])]
    return "\n".join(parts)

# ── 4.8  Token-budget trim ────────────────────────────────────────────────────
def _trim_to_token_budget(text: str, model, max_tokens: int) -> str:
    try:
        tokens=model.tokenize(text.encode("utf-8"))
    except Exception:
        lim=int(max_tokens*2.8); return text[:lim]+" [...]" if len(text)>lim else text
    if len(tokens)<=max_tokens: return text
    try:
        t=model.detokenize(tokens[:max_tokens]).decode("utf-8",errors="ignore")
    except Exception:
        t=text[:int(max_tokens*2.8)]
    lp=t.rfind(".")
    if lp>len(t)//2: t=t[:lp+1]
    return t+" [...]"

# ── 4.8b  Reasoning-trace stripper ───────────────────────────────────────────
def strip_reasoning(text: str) -> str:
    """Remove <think>…</think> CoT tags. No-op for current 5-model slate."""
    if not text: return text
    text=re.sub(r"<think>.*?</think>","",text,flags=re.DOTALL|re.IGNORECASE)
    text=re.sub(r"</?think>","",text,flags=re.IGNORECASE)
    return text.strip()

# ── 4.9  RAG + Graph inference ────────────────────────────────────────────────
def _clean_json_response(raw: str) -> str:
    raw=re.sub(r"<think>.*?</think>","",raw,flags=re.DOTALL|re.IGNORECASE)
    raw=re.sub(r"^```(?:json)?\s*","",raw,flags=re.MULTILINE)
    return re.sub(r"\s*```$","",raw,flags=re.MULTILINE).strip()

def _build_messages(system_content: str, user_content: str) -> list:
    """
    Build a chat messages list that respects each model's system role support.

    I-40 — Gemma-2 does not have a <system> token in its chat template.
    Google DeepMind designed the Gemma-2 instruction template with alternating
    [user, model] turns only; passing a "system" role raises:
      "System role not supported"
    The canonical workaround (Google Gemma cookbook, 2024; llama.cpp #7111)
    is to prepend the system instruction to the first user turn as plain text,
    separated by a blank line. The model receives identical instruction content;
    only the message boundary changes.

    All other models in MODEL_REGISTRY (Llama-3.1, Qwen2.5, Mistral-v0.3)
    support the system role natively — MODEL_SUPPORTS_SYSTEM=True for these.
    """
    if MODEL_SUPPORTS_SYSTEM:
        return [{"role":"system","content":system_content},
                {"role":"user",  "content":user_content}]
    else:
        # Gemma-2 pattern: fold system instruction into user turn
        return [{"role":"user",
                 "content":f"{system_content}\n\n{user_content}"}]

def perform_rag_legal_analysis(query, retriever, model,
                               knowledge_graph=None, nlp_model=None) -> tuple:
    _RAG_TOKEN_BUDGET=5_500
    docs=retriever.invoke(query)
    rag_ctx=_trim_to_token_budget("\n\n".join(d.page_content for d in docs),
                                  model,_RAG_TOKEN_BUDGET)
    # KG context (always active in v9)
    if knowledge_graph is not None and nlp_model is not None:
        g_ctx=graph_aware_retriever(query,knowledge_graph,nlp_model)
        g_fmt=_trim_to_token_budget(format_graph_context_for_llm(query,g_ctx),model,350)
    else:
        g_fmt=""
    prompt=(f"CONTEXTO RAG:\n{rag_ctx}\n\n"
            f"CONTEXTO GRAFO:\n{g_fmt}\n\n"
            f"CONSULTA:\n{query}\n\nANÁLISIS LEGAL:")
    _sys=("Eres un experto en derecho laboral mexicano. Responde con precisión "
          "técnica citando artículos de la LFT y jurisprudencia relevante.")
    full_text,token_count,_t="",0,_time.time()
    for ch in model.create_chat_completion(
            messages=_build_messages(_sys, prompt),
            max_tokens=512,temperature=0.1,stream=True):
        d=ch["choices"][0]["delta"].get("content","")
        if d: full_text+=d; token_count+=1
    print(f"\r    ✓  {token_count} tokens | "
          f"{token_count/max(_time.time()-_t,1e-6):.1f} tok/s     ",flush=True)
    analysis=(strip_reasoning(full_text) if MODEL_REASONING else full_text)
    return analysis,{"completion_tokens":token_count,"total_tokens":token_count}

def extract_case_to_json(analysis: str, model) -> dict:
    prompt=(f"Basado en el análisis legal, extrae la información en JSON:\n\n{analysis}\n\n"
            "Responde SOLO con este JSON. Para 'estado_del_procedimiento' usa ÚNICAMENTE "
            "una de estas tres opciones exactas: 'Procedente', 'Improcedente', "
            "'Parcialmente Improcedente'.\n"
            '{"partes_involucradas":[],"articulos_citados":[],'
            '"conclusiones_legales":[],"recomendaciones":[],'
            '"estado_del_procedimiento":"Procedente"}')
    resp=model.create_chat_completion(
        messages=_build_messages("Solo respondes con JSON válido.", prompt),
        max_tokens=512,temperature=0.0)
    try:
        return json.loads(_clean_json_response(resp["choices"][0]["message"]["content"]))
    except Exception:
        return {"error":"JSON parse failed"}

# ── 4.10  Metrics helpers ─────────────────────────────────────────────────────
def _flatten_articles(arts) -> list:
    if not arts: return []
    out=[]
    for it in arts:
        if isinstance(it,dict):
            out.append(" ".join(str(v) for v in it.values()
                                if v is not None and str(v).strip()))
        elif isinstance(it,list): out.extend(str(x) for x in it if x is not None)
        elif it is not None: out.append(str(it))
    return out

def normalise_decision(raw: str) -> str:
    import difflib as _dl
    if not raw or not isinstance(raw,str): return ""
    low=raw.lower().strip()
    _proc={"tramit","proceso","pendiente","análisis","revisión","estudio",
           "investigación","espera","curso","conocimiento","determinación",
           "resolución","diligencia"}
    if any(p in low for p in _proc): return ""
    if "parcialmente" in low: return "Parcialmente Improcedente"
    if "improcedente" in low or "no procede" in low or "no ha lugar" in low:
        return "Improcedente"
    if "procedente" in low or " procede" in low or "ha lugar" in low:
        return "Procedente"
    best=_dl.get_close_matches(raw,DECISION_LABELS,n=1,cutoff=0.5)
    return best[0] if best else ""

def evaluate_text_generation(predictions: list, references: list) -> pd.DataFrame:
    rouge=hf_evaluate.load("rouge").compute(predictions=predictions,references=references)
    bleu=hf_evaluate.load("bleu").compute(predictions=predictions,
                                          references=[[r] for r in references])
    try:
        _,_,F1=bert_score_compute(predictions,references,
                                   model_type="bert-base-multilingual-cased",
                                   num_layers=9,lang="es",verbose=False)
        bert_f1=round(float(F1.mean()),4)
    except Exception as e:
        print(f"  ⚠️  BERTScore failed ({e})."); bert_f1=0.0
    return pd.DataFrame([{"ROUGE-1":round(rouge["rouge1"],4),
                          "ROUGE-2":round(rouge["rouge2"],4),
                          "ROUGE-L":round(rouge["rougeL"],4),
                          "BLEU":round(bleu["bleu"],4),
                          "BERTScore-F1":bert_f1}])

def compute_article_hit_at_k(pred_list,gold_list) -> dict:
    P,R,F=[],[],[]
    for pred,gold in zip(pred_list,gold_list):
    # CORRECTIVE FIX: `.strip().lower()` compares raw surface forms, so a
    # correct citation written differently from gold scored as a miss.
    # Measured on the 60-case draft: raw matching gave article F1 of 0.005
    # (RAG) / 0.000 (GraphRAG) / 0.048 (RL) — the ordering reflected output
    # FORMAT, not legal grounding. Canonicalized: 0.134 / 0.159 / 0.207.
    # Uses the same shared normalizer as CrossModelCompar so in-notebook and
    # benchmark-table numbers cannot diverge.
        ps=normalize_articles(_flatten_articles(pred))
        gs=normalize_articles(_flatten_articles(gold))
        if not gs: continue
        tp=len(ps&gs); p=tp/len(ps) if ps else 0.0; r=tp/len(gs)
        P.append(p); R.append(r); F.append(2*p*r/(p+r) if(p+r) else 0.0)
    return {"article_precision":round(np.mean(P),4) if P else 0.0,
            "article_recall":round(np.mean(R),4) if R else 0.0,
            "article_F1":round(np.mean(F),4) if F else 0.0}

def compute_classification_metrics(y_true, y_pred, label_names=None,
                                   suffix=None):
    if label_names is None: label_names=DECISION_LABELS
    sfx=suffix or ARM_SUFFIX
    labels_ids=list(range(len(label_names)))
    print("\n  Classification Report:")
    print(classification_report(y_true,y_pred,labels=labels_ids,
                                target_names=label_names,zero_division=0))
    cm=confusion_matrix(y_true,y_pred,labels=labels_ids)
    fig,ax=plt.subplots(figsize=(7,5))
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",
                xticklabels=label_names,yticklabels=label_names,ax=ax)
    ax.set_title(f"Confusion Matrix — Legal Decision ({ARM})")
    plt.tight_layout()
    plt.savefig(f"/content/confusion_matrix_{sfx}.png",dpi=150); plt.show()

# ── 4.10b  Bootstrap confidence intervals ────────────────────────────────────
def bootstrap_prediction_cis(y_true, y_pred, n_boot=2000,
                              seed=42, alpha=0.05) -> dict:
    """95% percentile bootstrap CIs for accuracy, macro-F1, MCC."""
    y_true=np.asarray(y_true); y_pred=np.asarray(y_pred); n=len(y_true)
    if n==0:
        return {m:{"point":float("nan"),"lo":float("nan"),"hi":float("nan")}
                for m in("accuracy","macro_f1","mcc")}
    point={"accuracy":accuracy_score(y_true,y_pred),
           "macro_f1":f1_score(y_true,y_pred,average="macro",zero_division=0),
           "mcc":matthews_corrcoef(y_true,y_pred) if len(set(y_true))>1 else 0.0}
    rng=np.random.default_rng(seed); boot={m:[] for m in point}
    for _ in range(n_boot):
        idx=rng.integers(0,n,n); yt,yp=y_true[idx],y_pred[idx]
        boot["accuracy"].append(accuracy_score(yt,yp))
        boot["macro_f1"].append(f1_score(yt,yp,average="macro",zero_division=0))
        boot["mcc"].append(matthews_corrcoef(yt,yp) if len(set(yt))>1 else 0.0)
    out={}
    for m in point:
        lo,hi=np.percentile(boot[m],[100*alpha/2,100*(1-alpha/2)])
        out[m]={"point":round(float(point[m]),4),"lo":round(float(lo),4),
                "hi":round(float(hi),4)}
    return out

# ── 4.10c  _mrow: metric row with model metadata ─────────────────────────────
def _mrow(metric, value):
    return {"arm":ARM,"model":MODEL_KEY,"arch":MODEL_ARCH,"params_b":MODEL_PARAMS_B,
            "quant":MODEL_QUANT,"kg_enabled":int(KG_ENABLED),
            "metric":metric,"value":float(value)}

# ── 4.11  Litigation strategy ─────────────────────────────────────────────────

# CORRECTIVE FIX (post-hoc review, RL+LLM arm): a candidate model can produce
# syntactically valid, non-empty JSON that nonetheless abandons the required
# schema entirely (e.g. wrapping content under an unexpected container key,
# or omitting the decision/grounding fields). json.loads() succeeding is not
# evidence of schema compliance — only of syntactic validity. This validates
# the required keys are present before a parse is trusted; on failure the
# caller falls back to the same safe, correctly-keyed default dict already
# used for outright json.loads() failures.
_REQUIRED_STRATEGY_KEYS = ("fortalezas", "debilidades", "estrategia",
                           "articulos_clave", "probabilidad_exito")

def _is_valid_strategy_schema(d) -> bool:
    if not isinstance(d, dict):
        return False
    return all(k in d for k in _REQUIRED_STRATEGY_KEYS)


def litigation_strategy_analysis(facts, retriever, model,
                                  knowledge_graph=None, nlp_model=None) -> dict:
    docs=retriever.invoke(facts)
    rag_ctx=_trim_to_token_budget("\n\n".join(d.page_content for d in docs),model,4000)
    if knowledge_graph is not None and nlp_model is not None:
        g_fmt=_trim_to_token_budget(
            format_graph_context_for_llm(
                facts,graph_aware_retriever(facts,knowledge_graph,nlp_model)),
            model,300)
    else:
        g_fmt=""
    prompt=(f"CONTEXTO LEGAL (RAG + GRAFO DE CONOCIMIENTO):\n{rag_ctx}\n\n"
            f"ENTIDADES Y RELACIONES DEL GRAFO:\n{g_fmt}\n\n"
            f"HECHOS DEL CASO:\n{facts}\n\n"
            "Genera una estrategia de litigio ESTRUCTURADA en el siguiente formato JSON "
            "exacto. Para 'probabilidad_exito' usa ÚNICAMENTE: 'Alta', 'Media' o 'Baja'.\n\n"
            '{\n  "fortalezas": ["fortaleza legal 1", "fortaleza legal 2"],\n'
            '  "debilidades": ["debilidad o riesgo 1", "debilidad o riesgo 2"],\n'
            '  "estrategia": "descripción de la estrategia recomendada",\n'
            '  "articulos_clave": ["LFT art. 47", "LFT art. 48"],\n'
            '  "jurisprudencia_clave": ["tesis o jurisprudencia relevante"],\n'
            '  "probabilidad_exito": "Alta",\n'
            '  "razonamiento": "justificación jurídica detallada",\n'
            '  "advertencia_etica": "Esta predicción es orientativa y no sustituye '
            'el juicio de un abogado licenciado."\n}')
    raw=""
    _sys_lit=("Eres un abogado litigante experto en derecho laboral mexicano (LFT). "
              "Respondes ÚNICAMENTE con el JSON solicitado, sin texto adicional.")
    for ch in model.create_chat_completion(
            messages=_build_messages(_sys_lit, prompt),
            max_tokens=1024,temperature=0.1,stream=True):
        raw+=ch["choices"][0]["delta"].get("content","")
    try:
        parsed=json.loads(_clean_json_response(raw))
        if not _is_valid_strategy_schema(parsed):
            print(f"      ⚠️  strategy discarded: valid JSON but non-compliant "
                  f"schema (missing {[k for k in _REQUIRED_STRATEGY_KEYS if k not in parsed]}) "
                  f"— using safe fallback instead.")
            raise ValueError("schema non-compliant")
    except Exception:
        parsed={"fortalezas":[],"debilidades":[],"estrategia":raw,
                "articulos_clave":[],"jurisprudencia_clave":[],
                "probabilidad_exito":"","razonamiento":raw,
                "advertencia_etica":"Esta predicción es orientativa y no sustituye "
                                    "el juicio de un abogado licenciado."}
    parsed["raw_text"]=raw
    parsed["model"]=MODEL_KEY
    parsed["arm"]=ARM
    return parsed

# ── 4.12  Rubric ──────────────────────────────────────────────────────────────
def load_rubric(json_path: str) -> list:
    for p in [json_path,
              os.path.join(LOCAL_JSON,"rubrica_validacion_contratos.json"),
              "/content/rubrica_validacion_contratos.json"]:
        if os.path.isfile(p):
            with open(p,"r",encoding="utf-8") as f: data=json.load(f)
            q=data.get("questions",[])
            print(f"  ✅  Rubric: {len(q)} questions from '{p}'"); return q
    print("  ⚠️  Rubric JSON not found — evaluation skipped."); return []

def evaluate_rubric(strategy: dict, rubric_questions: list, model) -> list:
    results=[]
    s_txt=(f"Fortalezas: {'; '.join(strategy.get('fortalezas',[]))}\n"
           f"Debilidades: {'; '.join(strategy.get('debilidades',[]))}\n"
           f"Estrategia: {strategy.get('estrategia','')}\n"
           f"Artículos clave: {', '.join(strategy.get('articulos_clave',[]))}\n"
           f"Jurisprudencia: {', '.join(strategy.get('jurisprudencia_clave',[]))}\n"
           f"Probabilidad: {strategy.get('probabilidad_exito','')}\n"
           f"Razonamiento: {strategy.get('razonamiento','')}")
    for q in rubric_questions:
        crit=q["criteria"]
        crit_txt="\n".join(f"  Criterio '{c['name']}' (peso {c['weight_pct']}%): "
                           f"{c['guideline']}" for c in crit)
        jp=(f"Eres un abogado evaluador experto en derecho contractual mexicano.\n\n"
            f"PREGUNTA:\n{q['question']}\n\nRESPUESTA ESPERADA:\n{q['expected_answer']}\n\n"
            f"CRITERIOS:\n{crit_txt}\n\nESTRATEGIA A EVALUAR:\n{s_txt}\n\n"
            "Para cada criterio, determina si CUMPLE o NO CUMPLE. "
            "Responde ÚNICAMENTE con JSON:\n"
            +json.dumps({"question_id":q["id"],
                         "evaluations":[{"criterion":c["name"],"complies":True,
                                         "justification":"razón breve"}
                                        for c in crit]},ensure_ascii=False))
        raw=""
        try:
            for ch in model.create_chat_completion(
                    messages=_build_messages(
                        "Eres un evaluador legal preciso. Respondes SÓLO con JSON válido.",
                        jp),
                    max_tokens=512,temperature=0.0,stream=True):
                raw+=ch["choices"][0]["delta"].get("content","")
            evals=json.loads(_clean_json_response(raw)).get("evaluations",[])
        except Exception:
            evals=[{"criterion":c["name"],"complies":False,"justification":"Error"}
                   for c in crit]
        # CORRECTIVE FIX (post-hoc, RL+LLM arm's judge id-echo bug): matching by
        # exact string equality on the judge's ECHOED criterion name is fragile —
        # if the judge (Phi-3.5-mini) paraphrases, truncates, or re-accents the
        # label instead of reproducing it verbatim, the match silently fails and
        # a criterion the judge actually marked compliant gets scored 0. LLMs
        # preserve LIST ORDER far more reliably than they preserve exact string
        # labels, so we match evaluations to criteria POSITIONALLY instead —
        # the same fix already validated for the RL arm's grouped judge calls.
        cs=[]
        for _pos, c in enumerate(crit):
            ev = evals[_pos] if _pos < len(evals) else None
            comp=bool(ev.get("complies",False)) if ev else False
            cs.append({"criterion":c["name"],"weight_pct":c["weight_pct"],
                       "complies":comp,"score":c["weight_pct"] if comp else 0.0,
                       "justification":ev.get("justification","") if ev else ""})
        qsc=sum(x["score"] for x in cs)
        results.append({"question_id":q["id"],"question":q["question"],
                        "question_score":round(qsc,2),"criteria":cs})
        st="✅" if qsc>=66 else("⚠️ " if qsc>=33 else "❌")
        print(f"    {st}  Q{q['id']:02d}: {qsc:.1f}% — {q['question'][:55]}...")
    return results

def compute_rubric_summary(rr: list) -> dict:
    if not rr: return {}
    sc=[r["question_score"] for r in rr]; fp=sum(sc)/len(sc)
    wp=("Alta (≥80% cumplimiento)" if fp>=80 else
        "Media (60–79% cumplimiento)" if fp>=60 else "Baja (<60% cumplimiento)")
    return {"final_score_pct":round(fp,2),"win_probability":wp,
            "questions_passed":sum(1 for s in sc if s>=66),
            "questions_partial":sum(1 for s in sc if 33<=s<66),
            "questions_failed":sum(1 for s in sc if s<33),
            "question_scores":{f"Q{r['question_id']:02d}":r["question_score"] for r in rr}}

# ── 4.13  Inference-speed gate (PATCH A) ─────────────────────────────────────
def verify_inference_speed(model, min_tps=MIN_TPS, hard=True) -> float:
    """Probe throughput; abort (hard=True) or warn if below min_tps."""
    _t=_time.time()
    r=model.create_chat_completion(
        messages=[{"role":"user","content":"Enumera los números del 1 al 25."}],
        max_tokens=64,temperature=0.0)
    n=r.get("usage",{}).get("completion_tokens",64)
    tps=n/max(_time.time()-_t,1e-6)
    cuda=torch.cuda.is_available()
    print(f"  Speed probe: ~{tps:.1f} tok/s | torch.cuda={cuda}")
    if tps<min_tps:
        diag=("No GPU attached. Runtime → Change runtime type → T4 GPU." if not cuda
              else "GPU present but llama-cpp-python is a CPU build. Re-run §0.")
        banner=(f"\n{'='*70}\n  ❌  INFERENCE TOO SLOW: {tps:.1f} tok/s (< {min_tps}).\n"
                f"  §9 would take ~{60*(512/max(tps,0.1))/3600:.1f} h.\n  {diag}\n{'='*70}")
        if hard: raise RuntimeError(banner)
        print(banner)
    else:
        print(f"  ✅  GPU inference confirmed (~{tps:.1f} tok/s) → "
              f"§9 ETA ≈ {60*(512/tps)/60:.0f} min.")
    return tps

# ── 4.14  Checkpoint / resume (PATCH B) ──────────────────────────────────────
def _checkpoint_path() -> str:
    if RESUME and not os.path.exists(CHECKPOINT_LOCAL) and os.path.exists(CHECKPOINT_DRIVE):
        try:
            import shutil as _sh; _sh.copy2(CHECKPOINT_DRIVE,CHECKPOINT_LOCAL)
            print(f"  Restored checkpoint from Drive: {CHECKPOINT_DRIVE}")
        except Exception as e:
            print(f"  ⚠️  Could not restore Drive checkpoint: {e}")
    return CHECKPOINT_LOCAL

def load_checkpoint() -> list:
    p=_checkpoint_path()
    if not(RESUME and os.path.exists(p)): return []
    recs=[]
    with open(p,"r",encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line:
                try: recs.append(json.loads(line))
                except Exception: pass
    print(f"  ↻  Resuming: {len(recs)} cases already completed."); return recs

def append_checkpoint(rec: dict) -> None:
    with open(CHECKPOINT_LOCAL,"a",encoding="utf-8") as f:
        f.write(json.dumps(rec,ensure_ascii=False)+"\n")
    try:
        os.makedirs(os.path.dirname(CHECKPOINT_DRIVE),exist_ok=True)
        with open(CHECKPOINT_DRIVE,"a",encoding="utf-8") as f:
            f.write(json.dumps(rec,ensure_ascii=False)+"\n")
    except Exception:
        pass

print("✅  All helper functions defined.\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §5  PDF INGESTION
# ──────────────────────────────────────────────────────────────────────────────
print("── §5  PDF Ingestion ──────────────────────────────────────────────────")
raw_src=RAW_PDFS_PATH if os.path.exists(RAW_PDFS_PATH) else LOCAL_RAW_PDFS
law_src=LAW_PDFS_PATH if os.path.exists(LAW_PDFS_PATH) else LOCAL_LAW_PDFS
all_docs=load_pdfs_from_directory(raw_src)
law_docs=load_pdfs_from_directory(law_src)
print(f"  Pages — resolutions: {len(all_docs)} | laws: {len(law_docs)}\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §6  RAG VECTOR INDEX (FAISS) — shared across all 5 model runs
# ──────────────────────────────────────────────────────────────────────────────
print("── §6  RAG Vector Index ───────────────────────────────────────────────")
_faiss_local="/content/faiss_index"

# ── CORRECTIVE FIX (v13) · FAISS/embedder dimensionality guard ───────────────
# A cached index built with a DIFFERENT embedding model loads WITHOUT error and
# only fails later, deep inside faiss.search (`assert d == self.d`), surfacing
# as a bare AssertionError on every case with no usable message.
# OBSERVED IN PRODUCTION: a 384-dim index (MiniLM-family signature, 44,754
# vectors) persisted on Drive was silently reused against the 1024-dim
# intfloat/multilingual-e5-large embedder this pipeline specifies — all 60
# cases failed, 0 checkpointed. The load path below only rebuilt when NO cache
# existed, and never validated what it loaded.
# This guard makes the mismatch fail LOUDLY at load time, names both dims, and
# (when _FAISS_AUTO_REBUILD_ON_MISMATCH) moves the stale index aside so the
# rebuild branch runs instead of aborting the session.
# METHODOLOGICAL NOTE for the paper: the FAISS index is shared across all runs
# and, if also shared across arms, the embedding model is part of the frozen
# retrieval substrate. A stale index means documented methodology (e5-large)
# and actual behaviour (MiniLM) diverge — verify each arm's index.d before
# reporting cross-arm comparisons.
_FAISS_AUTO_REBUILD_ON_MISMATCH = True

def _assert_faiss_dim_matches(_db, _emb, _src_label):
    """Raise (or trigger rebuild) if a cached index's width != embedder width."""
    _expected = len(_emb.embed_query("dim probe"))
    _actual = _db.index.d
    if _actual == _expected:
        print(f"  ✅  dim check OK — index.d={_actual} == embedder dim={_expected}")
        return True
    msg = (f"FAISS index loaded from {_src_label} has dim={_actual}, but the "
           f"configured embedder ({EMBEDDING_MODEL_NAME if 'EMBEDDING_MODEL_NAME' in globals() else 'multilingual-e5-large'}) "
           f"produces dim={_expected}. The cached index was built with a "
           f"DIFFERENT embedding model and CANNOT be used — every retrieval "
           f"would fail inside faiss.search with a bare AssertionError.")
    if not _FAISS_AUTO_REBUILD_ON_MISMATCH:
        raise RuntimeError(msg + "  Set _FAISS_AUTO_REBUILD_ON_MISMATCH=True or "
                                 "move the stale index aside manually.")
    print(f"  ⚠️  {msg}")
    import time as _t, shutil as _sh
    _stamp = _t.strftime("%Y%m%d_%H%M%S")
    for _p in [_faiss_local, FAISS_INDEX_PATH]:
        if os.path.exists(_p):
            _bak = f"{_p}__STALE_{_actual}dim_{_stamp}"
            try:
                _sh.move(_p, _bak)
                print(f"     moved aside (kept for audit): {_p} → {_bak}")
            except Exception as _e:
                print(f"     ⚠️  could not move {_p}: {_e}")
    return False

_faiss_ok = False

# Load from cache (same session re-run or Drive-persisted from prior run).
# Each load is dim-checked; on mismatch the stale index is moved aside and
# _faiss_ok stays False so the rebuild branch below runs in the same pass.
if os.path.exists(_faiss_local):
    print(f"  Loading FAISS from local cache...")
    vector_db=FAISS.load_local(_faiss_local,embeddings,
                               allow_dangerous_deserialization=True)
    print(f"  ✅  {vector_db.index.ntotal:,} vectors (local).")
    _faiss_ok=_assert_faiss_dim_matches(vector_db,embeddings,"local cache")
elif os.path.exists(FAISS_INDEX_PATH):
    print(f"  Loading FAISS from Drive (shared with prior runs)...")
    vector_db=FAISS.load_local(FAISS_INDEX_PATH,embeddings,
                               allow_dangerous_deserialization=True)
    print(f"  ✅  {vector_db.index.ntotal:,} vectors (Drive).")
    _faiss_ok=_assert_faiss_dim_matches(vector_db,embeddings,"Drive")
    if _faiss_ok:
        vector_db.save_local(_faiss_local)

if not _faiss_ok:
    seps=["\n\n","\n","."," ",""]
    texts_prec=RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE_PREC,chunk_overlap=CHUNK_OVERLAP_PREC,
        separators=seps).split_documents(all_docs)
    texts_law=RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE_LAW,chunk_overlap=CHUNK_OVERLAP_LAW,
        separators=seps).split_documents(law_docs)
    combined=texts_prec+texts_law
    print(f"  Chunks — prec:{len(texts_prec):,} law:{len(texts_law):,} "
          f"total:{len(combined):,}")
    if not combined: raise RuntimeError("No chunks. Check PDF paths.")

    try:
        from tqdm.auto import tqdm as _tqdm; _HAS_TQDM=True
    except ImportError:
        _HAS_TQDM=False

    _BATCH=500; _t0=_time.time()
    _all_texts=[c.page_content for c in combined]
    print(f"  Building index (~{len(_all_texts)/70/60:.0f} min estimated)...")
    _vecs=[]
    _batches=range(0,len(_all_texts),_BATCH)
    if _HAS_TQDM: _batches=_tqdm(_batches,desc="Embedding",unit="batch")
    for _s in _batches:
        _vecs.extend(embeddings.embed_documents(_all_texts[_s:_s+_BATCH]))
        if not _HAS_TQDM:
            _d=min(_s+_BATCH,len(_all_texts)); _el=_time.time()-_t0
            _rt=_d/_el if _el else 0
            print(f"    {_d:>6,}/{len(_all_texts):,} | {_rt:>4.0f} seq/s | "
                  f"ETA {(_len_t:=len(_all_texts)-_d)/_rt/60 if _rt else 0:.1f} min")

    from langchain_community.docstore.in_memory import InMemoryDocstore
    import faiss as _fl
    _vn=np.array(_vecs,dtype="float32")
    _idx=_fl.IndexFlatIP(_vn.shape[1]); _idx.add(_vn)
    vector_db=FAISS(embedding_function=embeddings,index=_idx,
                    docstore=InMemoryDocstore({str(i):combined[i]
                                              for i in range(len(combined))}),
                    index_to_docstore_id={i:str(i) for i in range(len(combined))})
    print(f"  ✅  Built {_idx.ntotal:,} vectors in "
          f"{(_time.time()-_t0)/60:.1f} min")
    vector_db.save_local(_faiss_local)
    try:
        os.makedirs(FAISS_INDEX_PATH,exist_ok=True)
        vector_db.save_local(FAISS_INDEX_PATH)
        print(f"  ✅  Saved to Drive: {FAISS_INDEX_PATH}")
    except Exception as e:
        print(f"  ⚠️  Drive save skipped: {e}")


# ── FIX 3 · frozen-index assertion (HARD-FAIL) ───────────────────────────────
# The dim guard above catches an embedder mismatch but NOT "same dim, different
# chunking" — two 1024-dim indexes built from different corpora look identical
# to it. The manifest pins embedder + dim + ntotal + chunk parameters, and any
# divergence stops the run. Auto-rebuild is deliberately NOT offered here:
# silent rebuild is the mechanism that let the arms drift apart originally.
ensure_manifest(FAISS_INDEX_PATH, vector_db, embedder="intfloat/multilingual-e5-large",
                chunk_prec=CHUNK_SIZE_PREC, overlap_prec=CHUNK_OVERLAP_PREC,
                chunk_law=CHUNK_SIZE_LAW,   overlap_law=CHUNK_OVERLAP_LAW,
                arm=ARM, embed_fn=embeddings.embed_query)

retriever=vector_db.as_retriever(search_kwargs={"k":RETRIEVER_K})
print(f"✅  FAISS ready ({vector_db.index.ntotal:,} vectors, k={RETRIEVER_K}).\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §7  KNOWLEDGE GRAPH CONSTRUCTION — shared across all 5 model runs (R2)
# ──────────────────────────────────────────────────────────────────────────────
print("── §7  Knowledge Graph ────────────────────────────────────────────────")
_kg_local="/content/legal_kg.graphml"

if os.path.exists(_kg_local):
    print("  Loading KG from local cache...")
    legal_knowledge_graph=nx.read_graphml(_kg_local)
    print(f"  ✅  KG loaded: {len(legal_knowledge_graph.nodes()):,} nodes, "
          f"{len(legal_knowledge_graph.edges()):,} edges.")
elif os.path.exists(KG_DRIVE_PATH):
    print(f"  Loading KG from Drive: {KG_DRIVE_PATH}")
    legal_knowledge_graph=nx.read_graphml(KG_DRIVE_PATH)
    nx.write_graphml(legal_knowledge_graph,_kg_local)
    print(f"  ✅  KG loaded: {len(legal_knowledge_graph.nodes()):,} nodes, "
          f"{len(legal_knowledge_graph.edges()):,} edges.")
else:
    # Build from corpus — only on first run
    if not all_docs and not law_docs:
        print("  ⚠️  No PDFs loaded — empty KG.")
        legal_knowledge_graph=nx.DiGraph()
    else:
        # Use combined_texts from §6 if available, else rebuild chunks
        try:
            _chunk_src=combined
        except NameError:
            seps=["\n\n","\n","."," ",""]
            _t_p=RecursiveCharacterTextSplitter(
                chunk_size=CHUNK_SIZE_PREC,chunk_overlap=CHUNK_OVERLAP_PREC,
                separators=seps).split_documents(all_docs)
            _t_l=RecursiveCharacterTextSplitter(
                chunk_size=CHUNK_SIZE_LAW,chunk_overlap=CHUNK_OVERLAP_LAW,
                separators=seps).split_documents(law_docs)
            _chunk_src=_t_p+_t_l

        print(f"  Extracting entities from {len(_chunk_src):,} chunks...")
        all_entities,all_rels=[],[]
        _err={"unicode":0,"max_length":0,"other":0,"empty":0}
        _t7=_time.time()
        for idx,chunk in enumerate(_chunk_src):
            if idx>0 and idx%500==0:
                _el=_time.time()-_t7; _rt=idx/_el
                print(f"    {idx:>7,}/{len(_chunk_src):,} | {_rt:>4.0f} ch/s | "
                      f"ETA {(len(_chunk_src)-idx)/_rt/60:.1f} min | "
                      f"entities: {len(all_entities):,}",flush=True)
            try:
                info=extract_entities_and_relationships(chunk.page_content,nlp)
                all_entities.extend(info["entities"]); all_rels.extend(info["relationships"])
                continue
            except Exception as _e1: pass
            try:
                _asc=chunk.page_content.encode("ascii","ignore").decode("ascii").strip()
                if _asc:
                    info=extract_entities_and_relationships(_asc,nlp)
                    all_entities.extend(info["entities"]); all_rels.extend(info["relationships"])
                else: _err["empty"]+=1
            except Exception as _e2:
                _m=str(_e2)
                if any(k in _m.lower() for k in("integer","encode","unicode","surrogate")):
                    _err["unicode"]+=1
                else: _err["other"]+=1

        unique_ents=list({e["text"]:e for e in all_entities}.values())
        legal_knowledge_graph=build_knowledge_graph(unique_ents,all_rels)
        print(f"  ✅  KG built: {len(legal_knowledge_graph.nodes()):,} nodes, "
              f"{len(legal_knowledge_graph.edges()):,} edges | "
              f"time: {(_time.time()-_t7)/60:.1f} min | errors: {_err}")
        nx.write_graphml(legal_knowledge_graph,_kg_local)
        try:
            nx.write_graphml(legal_knowledge_graph,KG_DRIVE_PATH)
            print(f"  ✅  KG saved to Drive: {KG_DRIVE_PATH}")
        except Exception as e:
            print(f"  ⚠️  KG Drive save skipped: {e}")

print(f"✅  Knowledge Graph ready ({len(legal_knowledge_graph.nodes()):,} nodes).\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §8  BENCHMARK + RUBRIC LOADING
# ──────────────────────────────────────────────────────────────────────────────
print("── §8  Benchmark & Rubric Loading ────────────────────────────────────")
_eval_src=EVAL_JSON_PATH if os.path.exists(EVAL_JSON_PATH) else LOCAL_JSON
_gold_file=os.path.join(_eval_src,"gold_prediction_90_compatible.json")
test_cases=load_test_cases_gold(_gold_file if os.path.isfile(_gold_file) else _eval_src)
if not test_cases:
    print("  ⚠️  No cases loaded — using fallback demo case.")
    test_cases=[{"case_id":"DEMO-001",
        "query":"Trabajador demanda despido injustificado. El patrón no entregó aviso.",
        "ground_truth":"El despido es injustificado conforme art. 47 LFT.",
        "gold_decision":"Procedente","gold_articles":["LFT art. 47","LFT art. 48"],
        "confidence":"high","difficulty":"básico",
        "category":"Despido injustificado","critical":True,"precedent_refs":[]}]

rubric_questions=load_rubric(RUBRIC_JSON_PATH)
print(f"✅  {len(test_cases)} cases | {len(rubric_questions)} rubric questions.\n")


In [ ]:
# ── DIAGNOSTIC: does graph context actually reach the prompt? ────────────
import numpy as np
empty, n_nodes, n_edges, n_chars = 0, [], [], []

for case in test_cases:                       # all 90
    q = case.get("query", "")                 # ← flattened key used by §9
    if not q:
        continue
    ctx = graph_aware_retriever(q, legal_knowledge_graph, nlp)
    txt = format_graph_context_for_llm(q, ctx)
    trimmed = _trim_to_token_budget(txt, llm, 350) if txt else ""
    if not trimmed.strip():
        empty += 1
    n_nodes.append(len(ctx["nodes"])); n_edges.append(len(ctx["edges"]))
    n_chars.append(len(trimmed))

n = len(n_nodes)
print(f"cases with EMPTY graph context : {empty}/{n}")
print(f"nodes  per case: mean={np.mean(n_nodes):.1f}  median={np.median(n_nodes):.0f}  max={max(n_nodes)}")
print(f"edges  per case: mean={np.mean(n_edges):.1f}  median={np.median(n_edges):.0f}  max={max(n_edges)}")
print(f"chars in prompt: mean={np.mean(n_chars):.0f}  median={np.median(n_chars):.0f}  max={max(n_chars)}")

c = test_cases[0]
q = c.get("query", "")
doc = nlp(q)
ents = [(e.text, e.label_) for e in doc.ents]
kept = [e for e in ents if e[1] in LEGAL_ENTITY_TYPES]
print(f"\n{c.get('case_id')}: {len(ents)} entities, {len(kept)} pass the type filter")
print("  all   :", ents[:10])
print("  kept  :", kept[:10])
print("\n--- graph context as the model actually receives it ---")
print(_trim_to_token_budget(format_graph_context_for_llm(q,
      graph_aware_retriever(q, legal_knowledge_graph, nlp)), llm, 350)[:700] or "(EMPTY)")

In [ ]:
import numpy as np
echo, graph_part, total = [], [], []
for case in test_cases:
    q = case.get("query","")
    if not q: continue
    txt = format_graph_context_for_llm(q, graph_aware_retriever(q, legal_knowledge_graph, nlp))
    t = _trim_to_token_budget(txt, llm, 350) if txt else ""
    if not t.strip(): continue
    head = f"Consulta: {q}"
    e = len(head) if t.startswith("Consulta:") else 0
    echo.append(min(e, len(t))); graph_part.append(max(len(t)-e, 0)); total.append(len(t))

print(f"n non-empty            : {len(total)}")
print(f"mean chars total       : {np.mean(total):.0f}")
print(f"mean chars = query echo: {np.mean(echo):.0f}  ({100*np.mean(echo)/np.mean(total):.0f}%)")
print(f"mean chars = graph     : {np.mean(graph_part):.0f}  ({100*np.mean(graph_part)/np.mean(total):.0f}%)")
print(f"cases where graph part < 100 chars: {sum(1 for g in graph_part if g<100)}/{len(graph_part)}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §9  INFERENCE LOOP — GPU-gated, checkpoint/resume
# ──────────────────────────────────────────────────────────────────────────────
print("── §9  Inference Loop ─────────────────────────────────────────────────")

# PATCH A: hard GPU gate — abort before the 90-case loop if running on CPU
verify_inference_speed(llm, min_tps=MIN_TPS, hard=True)

# PATCH B: resume — rebuild accumulators, skip completed cases
all_preds,all_refs,all_pred_arts,all_gold_arts=[],[],[],[]
y_true_dec,y_pred_dec,results_rows,_loop_times=[],[],[],[]
_done_ids=set()
for _rec in load_checkpoint():
    all_preds.append(_rec["analysis"]); all_refs.append(_rec["ground_truth"])
    all_pred_arts.append(_rec["pred_arts"]); all_gold_arts.append(_rec["gold_arts"])
    _pl=DECISION_LABEL2ID.get(_rec["pred_dec"],-1)
    _gl=DECISION_LABEL2ID.get(_rec["gold_dec"],-1)
    if _gl!=-1 and _pl!=-1: y_true_dec.append(_gl); y_pred_dec.append(_pl)
    results_rows.append(_rec["row"]); _done_ids.add(_rec["case_id"])
_n_done=len(_done_ids)

print(f"  Running {len(test_cases)} cases "
      f"({_n_done} already done, {len(test_cases)-_n_done} remaining)...\n")

for i,case in enumerate(test_cases):
    cid=case.get("case_id",f"CASE-{i+1:03d}")
    query=case.get("query","")
    gold_txt=case.get("ground_truth","")
    gold_dec=case.get("gold_decision","")
    gold_arts=case.get("gold_articles",[])
    if cid in _done_ids: continue
    if not query or not gold_txt:
        print(f"  ⚠️  Skipping {cid}: missing query/ground_truth."); continue

    eta=(f" | ETA {(len(test_cases)-i)*sum(_loop_times)/len(_loop_times)/60:.0f} min"
         if _loop_times else "")
    print(f"  [{i+1:02d}/{len(test_cases)}] {cid} "
          f"({case.get('difficulty','?')}/{case.get('category','?')}){eta}")
    _t=_time.time()
    try:
        analysis,usage=perform_rag_legal_analysis(
            query,retriever,llm,
            knowledge_graph=legal_knowledge_graph,   # KG always active
            nlp_model=nlp)
        structured=extract_case_to_json(analysis,llm)
        pred_arts=_flatten_articles(structured.get("articulos_citados",[]))
        pred_dec_raw=structured.get("estado_del_procedimiento","")
        pred_dec=normalise_decision(pred_dec_raw)
        pl=DECISION_LABEL2ID.get(pred_dec,-1)
        gl=DECISION_LABEL2ID.get(gold_dec,-1)
        _sec=_time.time()-_t; _loop_times.append(_sec)

        row={"arm":ARM,"model":MODEL_KEY,"arch":MODEL_ARCH,"params_b":MODEL_PARAMS_B,
             "quant":MODEL_QUANT,"kg_enabled":int(KG_ENABLED),
             "case_id":cid,"category":case.get("category",""),
             "difficulty":case.get("difficulty",""),"critical":case.get("critical",False),
             "gold_decision":gold_dec,"pred_decision":pred_dec,
             "correct_decision":gold_dec==pred_dec,
             "n_gold_articles":len(gold_arts),"n_pred_articles":len(pred_arts),
             "prompt_tokens":usage.get("prompt_tokens",0),
             "completion_tokens":usage.get("completion_tokens",0),
             "inference_secs":round(_sec,1)}

        all_preds.append(analysis); all_refs.append(gold_txt)
        all_pred_arts.append(pred_arts); all_gold_arts.append(gold_arts)
        if gl!=-1 and pl!=-1: y_true_dec.append(gl); y_pred_dec.append(pl)
        results_rows.append(row); _done_ids.add(cid)

        # PATCH B: persist immediately (disconnect loses at most this one case)
        append_checkpoint({"case_id":cid,"analysis":analysis,
                           "ground_truth":gold_txt,"pred_arts":pred_arts,
                           "gold_arts":gold_arts,"pred_dec":pred_dec,
                           "gold_dec":gold_dec,"row":row})
        print(f"         Gold: {gold_dec:35s} Pred: {pred_dec}  [{_sec:.0f}s]")
    except Exception as e:
        # CORRECTIVE FIX: str(e) can be EMPTY for some exception types (bare
        # AssertionError, certain low-level ABI/SystemError failures) —
        # printing only {e} then gives zero diagnostic signal. Print the
        # type, repr, and traceback so the ACTUAL cause is always visible.
        import traceback as _tb
        print(f"  ❌  {cid}: [{type(e).__name__}] {e!r}  (will retry on next run)")
        _tb.print_exc(limit=6)

print(f"\n✅  Inference loop complete. {len(results_rows)} cases total "
      f"({_n_done} resumed, {len(results_rows)-_n_done} new).\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §10  EVALUATION — 3-tier metrics + MCC + bootstrap CIs
# ──────────────────────────────────────────────────────────────────────────────
print("── §10  Evaluation ────────────────────────────────────────────────────")

valid=[(p,r) for p,r in zip(all_preds,all_refs) if p and r]
gen_df=pd.DataFrame()
if valid:
    pl_,rl_=zip(*valid)
    print("\n  Computing ROUGE / BLEU / BERTScore...")
    gen_df=evaluate_text_generation(list(pl_),list(rl_))
    print(gen_df.to_string(index=False))

print("\n  Computing Article P/R/F1...")
art_metrics=compute_article_hit_at_k(all_pred_arts,all_gold_arts)
for k,v in art_metrics.items(): print(f"    {k}: {v}")

dec_acc=dec_macro_f1=dec_mcc=float("nan"); dec_cis={}
if y_true_dec and y_pred_dec:
    compute_classification_metrics(y_true_dec,y_pred_dec)
    dec_acc=accuracy_score(y_true_dec,y_pred_dec)
    dec_macro_f1=f1_score(y_true_dec,y_pred_dec,average="macro",zero_division=0)
    dec_mcc=(matthews_corrcoef(y_true_dec,y_pred_dec)
             if len(set(y_true_dec))>1 else 0.0)
    dec_cis=bootstrap_prediction_cis(y_true_dec,y_pred_dec)
    print(f"\n  Accuracy : {dec_acc:.4f}  "
          f"95% CI [{dec_cis['accuracy']['lo']}, {dec_cis['accuracy']['hi']}]")
    print(f"  Macro F1 : {dec_macro_f1:.4f}  "
          f"95% CI [{dec_cis['macro_f1']['lo']}, {dec_cis['macro_f1']['hi']}]")
    print(f"  MCC      : {dec_mcc:.4f}  "
          f"95% CI [{dec_cis['mcc']['lo']}, {dec_cis['mcc']['hi']}]")

results_df=pd.DataFrame(results_rows)
if not results_df.empty:
    _csv=f"/content/benchmark_results_{ARM_SUFFIX}.csv"
    results_df.to_csv(_csv,index=False,encoding="utf-8")
    n_ok,n_tot=results_df["correct_decision"].sum(),len(results_df)
    print(f"\n  Decision accuracy: {n_ok}/{n_tot} ({100*n_ok/n_tot:.1f}%)")
    print(f"  Saved: {_csv}")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §11  LITIGATION STRATEGY — structured RAG + KG output
# ──────────────────────────────────────────────────────────────────────────────
print("\n── §11  Litigation Strategy ───────────────────────────────────────────")
_demo_facts=(
    "Un empleado con 5 años de antigüedad fue despedido verbalmente por su "
    "supervisor sin causa justificada. El patrón no entregó aviso escrito de "
    "rescisión, no invocó artículos de la LFT y no pagó partes proporcionales. "
    "El trabajador demanda indemnización constitucional, salarios caídos y "
    "prima de antigüedad conforme a la Ley Federal del Trabajo.")
print(f"  Facts: {_demo_facts[:100]}...\n")

litigation_result=litigation_strategy_analysis(
    _demo_facts,retriever,llm,
    knowledge_graph=legal_knowledge_graph,   # KG active
    nlp_model=nlp)

print(f"\n  Probabilidad de éxito : {litigation_result.get('probabilidad_exito','')}")
print(f"  Artículos clave       : "
      f"{', '.join(litigation_result.get('articulos_clave',[]))}")
print("  Fortalezas:")
[print(f"    • {x}") for x in litigation_result.get("fortalezas",[])]
print("  Debilidades:")
[print(f"    • {x}") for x in litigation_result.get("debilidades",[])]
print(f"  Estrategia: {litigation_result.get('estrategia','')[:200]}")
print(f"  ⚠️  {litigation_result.get('advertencia_etica','')}")

_lit=f"/content/litigation_strategy_{ARM_SUFFIX}.json"
with open(_lit,"w",encoding="utf-8") as f:
    json.dump({"arm":ARM,"model":MODEL_KEY,"kg_enabled":KG_ENABLED,
               "demo_facts":_demo_facts,
               "litigation_strategy":litigation_result},f,indent=2,ensure_ascii=False)
print(f"\n  ✅  Saved: {_lit}")


# ──────────────────────────────────────────────────────────────────────────────
# §11b  RUBRIC EVALUATION — 20-question LLM-as-judge
# ──────────────────────────────────────────────────────────────────────────────
print("\n── §11b  Rubric Evaluation ────────────────────────────────────────────")
rubric_results,rubric_summary=[],{}
if rubric_questions:
    print(f"  Evaluating against {len(rubric_questions)} questions...\n")
    _judge_llm=judge_mgr.get()  # loads on demand; not resident during generation
    rubric_results=evaluate_rubric(litigation_result,rubric_questions,_judge_llm)  # FIX: fixed judge, not candidate
    judge_mgr.release()
    rubric_summary=compute_rubric_summary(rubric_results)
    print(f"\n  Final score: {rubric_summary.get('final_score_pct',0):.1f}%  | "
          f"{rubric_summary.get('win_probability','')}")
    _rj=f"/content/rubric_evaluation_results_{ARM_SUFFIX}.json"
    with open(_rj,"w",encoding="utf-8") as f:
        json.dump({"arm":ARM,"model":MODEL_KEY,"arch":MODEL_ARCH,
                   "quant":MODEL_QUANT,"kg_enabled":KG_ENABLED,
                   "judge_model":JUDGE_KEY,
                   "summary":rubric_summary,"results":rubric_results},
                  f,indent=2,ensure_ascii=False)
    print(f"  ✅  Saved: {_rj}")
else:
    print("  Rubric skipped (no questions loaded).")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §12  VISUALISATION + §12b QUALITATIVE SAMPLE
# ──────────────────────────────────────────────────────────────────────────────
print("\n── §12  Visualisation & Export ────────────────────────────────────────")

# ── 12.1  Benchmark: accuracy by difficulty + decision distribution ───────────
if not results_df.empty:
    fig,axes=plt.subplots(1,2,figsize=(13,4))
    acc_by=results_df.groupby("difficulty")["correct_decision"]\
        .agg(["mean","count"]).reset_index()
    axes[0].bar(acc_by["difficulty"],acc_by["mean"],color="#9DC3E6",edgecolor="#1F3864")
    axes[0].set_ylim(0,1.05); axes[0].set_title(f"Decision Accuracy by Difficulty ({ARM})")
    for j,row in acc_by.iterrows():
        axes[0].text(j,row["mean"]+0.02,f"n={int(row['count'])}",
                     ha="center",fontsize=9)
    gc=results_df["gold_decision"].value_counts()
    pc=results_df["pred_decision"].value_counts()
    x=np.arange(len(DECISION_LABELS)); w=0.35
    axes[1].bar(x-w/2,[gc.get(l,0) for l in DECISION_LABELS],w,
                label="Gold",color="#1F3864")
    axes[1].bar(x+w/2,[pc.get(l,0) for l in DECISION_LABELS],w,
                label="Predicted",color="#9DC3E6")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([l[:15] for l in DECISION_LABELS],rotation=10)
    axes[1].set_title("Decision Distribution"); axes[1].legend()
    plt.suptitle(f"Graph RAG + LLM Legal v9 — [{ARM}]",
                 fontsize=13,fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"/content/benchmark_summary_plots_{ARM_SUFFIX}.png",dpi=150)
    plt.show()

# ── 12.2  Rubric bar chart ────────────────────────────────────────────────────
if rubric_results:
    _q_ids=[f"Q{r['question_id']:02d}" for r in rubric_results]
    _q_sc=[r["question_score"] for r in rubric_results]
    _colors=["#375623" if s>=66 else "#7F6000" if s>=33 else "#C00000" for s in _q_sc]
    fig2,ax2=plt.subplots(figsize=(16,5))
    bars=ax2.bar(_q_ids,_q_sc,color=_colors,edgecolor="white",linewidth=0.5)
    ax2.axhline(66,color="#375623",linestyle="--",linewidth=1.2,label="Cumple (66%)")
    ax2.axhline(33,color="#7F6000",linestyle=":",linewidth=1.2,label="Parcial (33%)")
    ax2.axhline(rubric_summary.get("final_score_pct",0),
                color="#1F3864",linestyle="-",linewidth=2,
                label=f"Score Final: {rubric_summary.get('final_score_pct',0):.1f}%")
    for bar,sc in zip(bars,_q_sc):
        ax2.text(bar.get_x()+bar.get_width()/2,bar.get_height()+1.5,
                 f"{sc:.0f}%",ha="center",va="bottom",fontsize=7,fontweight="bold")
    ax2.set_ylim(0,115)
    ax2.set_title(f"Rúbrica de Validación [{ARM}]\n"
                  f"Score: {rubric_summary.get('final_score_pct',0):.1f}%  |  "
                  f"Prob. éxito: {rubric_summary.get('win_probability','')}",
                  fontsize=11,fontweight="bold")
    ax2.legend(fontsize=9); plt.xticks(rotation=45,ha="right",fontsize=8)
    plt.tight_layout()
    plt.savefig(f"/content/rubric_evaluation_chart_{ARM_SUFFIX}.png",dpi=150)
    plt.show()

# ── 12.3  Factor-tagged metrics summary ──────────────────────────────────────
_metrics_rows=[]
if not gen_df.empty:
    for col in gen_df.columns:
        _metrics_rows.append(_mrow(col,gen_df.iloc[0][col]))
for k,v in art_metrics.items():
    _metrics_rows.append(_mrow(k,v))
_metrics_rows.append(_mrow("decision_accuracy",dec_acc))
_metrics_rows.append(_mrow("decision_macro_f1",dec_macro_f1))
_metrics_rows.append(_mrow("decision_mcc",dec_mcc))
if dec_cis:
    for _m,_key in[("accuracy","decision_accuracy"),
                   ("macro_f1","decision_macro_f1"),("mcc","decision_mcc")]:
        _metrics_rows.append(_mrow(f"{_key}_ci_lo",dec_cis[_m]["lo"]))
        _metrics_rows.append(_mrow(f"{_key}_ci_hi",dec_cis[_m]["hi"]))
# Per-class P/R/F1 (R5)
if y_true_dec and y_pred_dec:
    from sklearn.metrics import precision_recall_fscore_support
    _pc_p,_pc_r,_pc_f,_pc_s=precision_recall_fscore_support(
        y_true_dec,y_pred_dec,labels=list(range(len(DECISION_LABELS))),
        zero_division=0)
    for _ci,_lbl in enumerate(DECISION_LABELS):
        _slug=_lbl.lower().replace(" ","_")
        _metrics_rows.append(_mrow(f"precision_{_slug}",_pc_p[_ci]))
        _metrics_rows.append(_mrow(f"recall_{_slug}",   _pc_r[_ci]))
        _metrics_rows.append(_mrow(f"f1_{_slug}",       _pc_f[_ci]))
        _metrics_rows.append(_mrow(f"support_{_slug}",  int(_pc_s[_ci])))
if rubric_summary:
    _metrics_rows.append(_mrow("rubric_final_score_pct",
                               rubric_summary.get("final_score_pct",0)))
    _metrics_rows.append(_mrow("rubric_questions_passed",
                               rubric_summary.get("questions_passed",0)))

metrics_summary_df=pd.DataFrame(_metrics_rows)
_msum=f"/content/metrics_summary_{ARM_SUFFIX}.csv"
metrics_summary_df.to_csv(_msum,index=False,encoding="utf-8")
print(metrics_summary_df[["metric","value"]].to_string(index=False))
print(f"  ✅  Saved: {_msum}")

# ── 12.4  KG statistics ───────────────────────────────────────────────────────
kg_stats={"nodes":len(legal_knowledge_graph.nodes()),
          "edges":len(legal_knowledge_graph.edges()),
          "entity_types":len(set(
              nx.get_node_attributes(legal_knowledge_graph,"label").values())),
          "rel_types":len(set(
              nx.get_edge_attributes(legal_knowledge_graph,"type").values()))}
print("\n  KG statistics:")
for k,v in kg_stats.items(): print(f"    {k}: {v}")

# ── §12b  Qualitative 5-case sample (thesis narrative) ───────────────────────
print("\n── §12b  Qualitative 5-case sample ───────────────────────────────────")
_qual=[]
try:
    _recs=load_checkpoint()
    _seen_cat,_picked=set(),[]
    for _r in sorted(_recs,key=lambda r:(r["row"].get("difficulty",""),
                                         r["row"].get("category",""))):
        _cat=_r["row"].get("category","")
        if _cat not in _seen_cat: _seen_cat.add(_cat); _picked.append(_r)
        if len(_picked)>=5: break
    if len(_picked)<5:
        _picked+=[r for r in _recs if r not in _picked][:5-len(_picked)]
    for _r in _picked[:5]:
        _qual.append({"case_id":_r["case_id"],
                      "category":_r["row"].get("category",""),
                      "difficulty":_r["row"].get("difficulty",""),
                      "gold_decision":_r["gold_dec"],
                      "pred_decision":_r["pred_dec"],
                      "correct":_r["gold_dec"]==_r["pred_dec"],
                      "gold_articles":_r["gold_arts"],
                      "pred_articles":_r["pred_arts"],
                      "analysis":_r["analysis"]})
    _qf=f"/content/qualitative_sample_{ARM_SUFFIX}.json"
    with open(_qf,"w",encoding="utf-8") as f:
        json.dump({"model":MODEL_KEY,"arch":MODEL_ARCH,"quant":MODEL_QUANT,
                   "kg_enabled":KG_ENABLED,"arm":ARM,"cases":_qual},
                  f,indent=2,ensure_ascii=False)
    print(f"  ✅  {len(_qual)} representative cases → {_qf}")
except Exception as e:
    print(f"  ⚠️  Qualitative sample skipped: {e}")

print(f"\n✅  Pipeline complete.")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §13  SAVE ALL ARTIFACTS TO DRIVE — ARM-namespaced subdirectory
# ──────────────────────────────────────────────────────────────────────────────
print("\n── §13  Saving Artifacts to Drive ────────────────────────────────────")
import shutil as _sh
_OUT=os.path.join(DRIVE_ROOT,"results",ARM)
os.makedirs(_OUT,exist_ok=True)
_saved,_failed=[],[]

for fn in [
    f"benchmark_results_{ARM_SUFFIX}.csv",
    f"confusion_matrix_{ARM_SUFFIX}.png",
    f"benchmark_summary_plots_{ARM_SUFFIX}.png",
    f"metrics_summary_{ARM_SUFFIX}.csv",
    f"litigation_strategy_{ARM_SUFFIX}.json",
    f"rubric_evaluation_results_{ARM_SUFFIX}.json",
    f"rubric_evaluation_chart_{ARM_SUFFIX}.png",
    f"qualitative_sample_{ARM_SUFFIX}.json",
    f"checkpoint_{ARM_SUFFIX}.jsonl",
]:
    src=f"/content/{fn}"
    if os.path.exists(src):
        try:
            _sh.copy2(src,os.path.join(_OUT,fn)); _saved.append(fn)
        except Exception as e:
            _failed.append((fn,str(e)))

# Also persist KG and FAISS (shared, already on Drive — copy only if local newer)
for _src,_dst in[(_kg_local, KG_DRIVE_PATH),
                  (_faiss_local, FAISS_INDEX_PATH)]:
    if os.path.exists(_src) and not os.path.exists(_dst):
        try:
            if os.path.isdir(_src):
                import shutil; shutil.copytree(_src,_dst)
            else:
                _sh.copy2(_src,_dst)
            _saved.append(os.path.basename(_dst))
        except Exception as e:
            _failed.append((os.path.basename(_dst),str(e)))

print(f"\n  Saved to {_OUT}:")
for fn in _saved: print(f"    ✅  {fn}")
if _failed:
    print(f"\n  Failed:")
    for fn,err in _failed: print(f"    ❌  {fn}: {err}")
print(f"\n✅  §13 complete.\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §14  CASE ANALYSIS MODULE — Reusable PhD Experiment API
# ──────────────────────────────────────────────────────────────────────────────
# PURPOSE: run additional case analyses (beyond the §11 demo and §9 benchmark)
# without modifying any prior section. Call from new Colab cells.
#
# All pipeline objects (llm, retriever, legal_knowledge_graph, nlp,
# rubric_questions) are already in memory. Results are tagged with MODEL_KEY
# so multiple model runs remain distinguishable in the registry and on Drive.
#
# USAGE (new cell, no script changes):
#   facts = "Trabajadora embarazada despedida durante licencia de maternidad."
#   result = run_case_analysis("CASE-MAT-01", facts)
#   compare_cases()
#   export_case_report("CASE-MAT-01")

import datetime as _dt

_CASE_REGISTRY: dict = {}
_CASES_DIR = os.path.join(DRIVE_ROOT,"case_analyses",MODEL_KEY)
os.makedirs(_CASES_DIR,exist_ok=True)


def run_case_analysis(case_id: str, facts: str,
                      save: bool=True, verbose: bool=True) -> dict:
    """
    Full litigation analysis + rubric evaluation on any case scenario.

    Parameters
    ──────────
    case_id : str  — unique ID, used as file stem in Drive export
    facts   : str  — case scenario in Spanish
    save    : bool — persist to Drive/case_analyses/{MODEL_KEY}/{case_id}.json
    verbose : bool — print formatted results

    Returns dict with keys: case_id, facts, strategy, rubric, summary, model, timestamp
    """
    if not facts or not facts.strip():
        raise ValueError("facts cannot be empty.")
    if verbose:
        print(f"\n{'═'*70}")
        print(f"  CASE ANALYSIS: {case_id}  [{MODEL_KEY}]")
        print(f"{'═'*70}")
        print(f"  Facts: {facts[:120]}{'...' if len(facts)>120 else ''}\n")

    # Step 1: Litigation strategy (RAG + KG)
    if verbose: print("  [1/2] Generating litigation strategy (RAG + KG)...")
    strategy=litigation_strategy_analysis(
        facts,retriever,llm,
        knowledge_graph=legal_knowledge_graph,
        nlp_model=nlp)
    if verbose:
        print(f"\n  Probabilidad de éxito : {strategy.get('probabilidad_exito','—')}")
        print(f"  Artículos clave       : "
              f"{', '.join(strategy.get('articulos_clave',[]))[:80] or '—'}")
        print("  Fortalezas:")
        [print(f"    • {x}") for x in strategy.get("fortalezas",[])]
        print("  Debilidades:")
        [print(f"    • {x}") for x in strategy.get("debilidades",[])]
        print(f"  Estrategia: {strategy.get('estrategia','')[:200]}")

    # Step 2: Rubric evaluation
    rubric,summary=[],{}
    if rubric_questions:
        if verbose: print(f"\n  [2/2] Evaluating against {len(rubric_questions)}-Q rubric...")
        _judge_llm=judge_mgr.get()  # loads on demand; not resident during generation
        rubric=evaluate_rubric(strategy,rubric_questions,_judge_llm)  # FIX: fixed judge, not candidate
        judge_mgr.release()
        summary=compute_rubric_summary(rubric)
        if verbose:
            print(f"\n  Score: {summary.get('final_score_pct',0):.1f}% | "
                  f"{summary.get('win_probability','—')}")
            print(f"  Questions passed: {summary.get('questions_passed',0)}/20")
    else:
        if verbose: print("  [2/2] Rubric not loaded — skipped.")

    if verbose:
        print(f"\n  ⚠️  {strategy.get('advertencia_etica','')}")
        print(f"{'═'*70}\n")

    result={"case_id":case_id,"facts":facts,"strategy":strategy,
            "rubric":rubric,"summary":summary,
            "model":MODEL_KEY,"arm":ARM,"kg_enabled":KG_ENABLED,
            "timestamp":_dt.datetime.now().isoformat()}

    _CASE_REGISTRY[case_id]=result

    if save:
        _out=os.path.join(_CASES_DIR,f"{case_id}.json")
        try:
            with open(_out,"w",encoding="utf-8") as f:
                json.dump(result,f,indent=2,ensure_ascii=False)
            if verbose: print(f"  ✅  Saved: {_out}")
        except Exception as e:
            if verbose: print(f"  ⚠️  Drive save failed: {e}")

    return result


def compare_cases(case_ids: list=None) -> None:
    """
    Side-by-side comparison table of all analysed cases.
    Automatically loads Drive-saved cases from previous sessions.
    """
    ids=case_ids if case_ids else list(_CASE_REGISTRY.keys())

    # Load Drive-saved cases not in current session registry
    if os.path.isdir(_CASES_DIR):
        for _fn in sorted(os.listdir(_CASES_DIR)):
            if _fn.endswith(".json"):
                _cid=_fn[:-5]
                if _cid not in _CASE_REGISTRY:
                    try:
                        with open(os.path.join(_CASES_DIR,_fn),
                                  encoding="utf-8") as _fp:
                            _CASE_REGISTRY[_cid]=json.load(_fp)
                        if case_ids is None: ids.append(_cid)
                    except Exception: pass

    if not ids: print("No cases in registry. Run run_case_analysis() first."); return
    _COL=22; _SEP="─"*(8+len(ids)*(_COL+3))
    print(f"\n{'═'*(8+len(ids)*(_COL+3))}")
    print(f"  CASE COMPARISON  [{MODEL_KEY}]  ({len(ids)} cases)")
    print(_SEP)
    _hdr=f"  {'Metric':<32}"
    for cid in ids: _hdr+=f"  {cid[:_COL]:<{_COL}}"
    print(_hdr); print(_SEP)

    def _row(label,fn):
        _line=f"  {label:<32}"
        for cid in ids:
            rec=_CASE_REGISTRY.get(cid,{})
            try: val=str(fn(rec))[:_COL]
            except Exception: val="—"
            _line+=f"  {val:<{_COL}}"
        print(_line)

    _row("Prob. éxito",lambda r:r.get("strategy",{}).get("probabilidad_exito","—"))
    _row("Score rúbrica (%)",lambda r:f"{r.get('summary',{}).get('final_score_pct',0):.1f}%")
    _row("Preguntas aprobadas (/20)",lambda r:f"{r.get('summary',{}).get('questions_passed',0)}/20")
    _row("Artículos clave",lambda r:", ".join(r.get("strategy",{}).get("articulos_clave",[]))[:_COL])
    _row("Modelo",lambda r:r.get("model","—"))
    _row("Timestamp",lambda r:r.get("timestamp","—")[:19])
    print(_SEP)
    _prob_line=f"  {'Prob. éxito (icono)':<32}"
    for cid in ids:
        prob=_CASE_REGISTRY.get(cid,{}).get("strategy",{}).get("probabilidad_exito","")
        icon="🟢 Alta " if prob=="Alta" else("🟡 Media" if prob=="Media" else "🔴 Baja ")
        _prob_line+=f"  {icon:<{_COL}}"
    print(_prob_line)
    print(f"{'═'*(8+len(ids)*(_COL+3))}\n")


def export_case_report(case_id: str) -> str:
    """Re-export a formatted report for a specific case to Drive."""
    if case_id not in _CASE_REGISTRY:
        _path=os.path.join(_CASES_DIR,f"{case_id}.json")
        if os.path.exists(_path):
            with open(_path,encoding="utf-8") as f:
                _CASE_REGISTRY[case_id]=json.load(f)
        else:
            raise FileNotFoundError(
                f"Case '{case_id}' not found.\n"
                f"Run run_case_analysis('{case_id}', facts) first.")
    _out=os.path.join(_CASES_DIR,f"{case_id}_report.json")
    with open(_out,"w",encoding="utf-8") as f:
        json.dump(_CASE_REGISTRY[case_id],f,indent=2,ensure_ascii=False)
    print(f"✅  Report exported: {_out}")
    return _out


# ── Module ready ──────────────────────────────────────────────────────────────
print("\n"+"═"*70)
print(f"  §14  CASE ANALYSIS MODULE READY  [{MODEL_KEY} | {ARM}]")
print("═"*70)
print(f"""
  Functions (call from any new cell after §14 has run):
  ───────────────────────────────────────────────────────
  run_case_analysis(case_id, facts, save=True, verbose=True)
    → Litigation strategy + 20-Q rubric evaluation.
    → Saved to Drive: SCJN_RAG/case_analyses/{MODEL_KEY}/{{case_id}}.json

  compare_cases(case_ids=None)
    → Side-by-side table of all cases in this session + Drive-saved.

  export_case_report(case_id)
    → Re-export a formatted report for one case.

  Example (paste in a new cell):
  ───────────────────────────────────────────────────────
  facts = (
      "Trabajadora con 3 años de antigüedad es despedida durante "
      "su licencia de maternidad. El patrón alega reorganización."
  )
  result = run_case_analysis("CASE-MAT-01", facts)
""")
print("═"*70)